In [1]:
import sys

print("Python do notebook:")
print(sys.executable)

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm

print("\nReportLab OK")

Python do notebook:
C:\Users\beelt\Documents\collections_case_candidate\.venv\Scripts\python.exe

ReportLab OK


In [2]:
from pathlib import Path
import json
import io
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle,
    PageBreak,
    Image,
)

warnings.filterwarnings("ignore")


In [3]:
ROOT = Path.cwd()

CANDIDATES = [
    ROOT / "05_bivariate_eda_collections_macro_analysis.ipynb",
    ROOT.parent / "05_bivariate_eda_collections_macro_analysis.ipynb",
    ROOT / "notebooks" / "05_bivariate_eda_collections_macro_analysis.ipynb",
]

BIVARIATE_NOTEBOOK = next((p for p in CANDIDATES if p.exists()), None)

if BIVARIATE_NOTEBOOK is None:
    raise FileNotFoundError(
        "Não encontrei 05_bivariate_eda_collections_macro_analysis.ipynb. "
        "Ajuste BIVARIATE_NOTEBOOK nesta célula."
    )

REPORT_DIR = ROOT / "reports"
CHART_DIR = REPORT_DIR / "collections_report_charts"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
CHART_DIR.mkdir(parents=True, exist_ok=True)

PDF_PATH = REPORT_DIR / "collections_macro_bivariate_analysis_report.pdf"

print("Notebook fonte :", BIVARIATE_NOTEBOOK.resolve())
print("PDF de saída   :", PDF_PATH.resolve())


Notebook fonte : C:\Users\beelt\Documents\collections_case_candidate\notebooks\05_bivariate_eda_collections_macro_analysis.ipynb
PDF de saída   : C:\Users\beelt\Documents\collections_case_candidate\notebooks\reports\collections_macro_bivariate_analysis_report.pdf


In [4]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_DIR = Path("../data")
QUEUE_PATH = DATA_DIR / "raw" / "collections_queue_sep2026.csv"
WA_PATH = DATA_DIR / "raw" / "whatsapp_collections_history.csv"

if not QUEUE_PATH.exists():
    QUEUE_PATH = Path("/mnt/data/collections_queue_sep2026(1).csv")
if not WA_PATH.exists():
    WA_PATH = Path("/mnt/data/whatsapp_collections_history(2).csv")

queue = pd.read_csv(QUEUE_PATH)
wa = pd.read_csv(WA_PATH)

wa["sent_at"] = pd.to_datetime(wa["sent_at"], errors="coerce")
queue["in_collections_since"] = pd.to_datetime(queue["in_collections_since"], errors="coerce")

print("Queue:", queue.shape)
print("WhatsApp:", wa.shape)

Queue: (10658, 10)
WhatsApp: (75406, 17)


In [5]:
##_______________________________________________________________ "" ____________________________________________________________

In [6]:
# ============================================================
# DEEP ANALYSIS
# FIRST CONTACT × DPD × APP LOGIN × PAYMENT
#
# Question:
# Does recent app activity help identify the right timing
# for the FIRST collections message, beyond DPD?
#
# Population:
#   First observed WhatsApp per customer
#   DPD 1–30
#
# Outcomes:
#   Payment within 72h
#   Recovery / message
# ============================================================

import numpy as np
import pandas as pd

# ============================================================
# 1. BASE
# ============================================================

x = wa.copy()

x["sent_at"] = pd.to_datetime(x["sent_at"])

x["days_past_due"] = pd.to_numeric(
    x["days_past_due"],
    errors="coerce"
)

x["days_since_last_app_login"] = pd.to_numeric(
    x["days_since_last_app_login"],
    errors="coerce"
)

x["amount_paid_brl"] = pd.to_numeric(
    x["amount_paid_brl"],
    errors="coerce"
).fillna(0)

x["payment_72h"] = (
    x["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

x["recovery_72h"] = np.where(
    x["payment_72h"],
    x["amount_paid_brl"],
    0
)


# ============================================================
# 2. FIRST OBSERVED MESSAGE PER CUSTOMER
# ============================================================

first = (
    x.sort_values(
        ["customer_id", "sent_at", "message_id"]
    )
    .groupby("customer_id", as_index=False)
    .first()
)

# Early collections only
first = first.loc[
    first["days_past_due"].between(1, 30)
].copy()


print("=" * 100)
print("FIRST CONTACT — EARLY COLLECTIONS")
print("=" * 100)

print(f"Customers/messages : {len(first):,}")
print(
    f"Payment events     : "
    f"{first['payment_72h'].sum():,}"
)

print(
    f"Payment rate       : "
    f"{first['payment_72h'].mean():.2%}"
)

print(
    f"Recovery           : "
    f"R$ {first['recovery_72h'].sum():,.2f}"
)

print(
    f"Recovery / message : "
    f"R$ {first['recovery_72h'].mean():,.2f}"
)

print(
    f"Missing app login  : "
    f"{first['days_since_last_app_login'].isna().sum():,}"
)


# ============================================================
# 3. DISTRIBUTION OF APP LOGIN RECENCY
# ============================================================

print("\n" + "=" * 100)
print("APP LOGIN RECENCY — DISTRIBUTION")
print("=" * 100)

display(
    first["days_since_last_app_login"]
    .describe(
        percentiles=[
            .01, .05, .10, .25,
            .50, .75, .90, .95, .99
        ]
    )
    .to_frame("days_since_last_app_login")
)


# ============================================================
# 4. CREATE LOGIN BUCKETS
#
# Keep relatively granular initially.
# We can consolidate after seeing N.
# ============================================================

first["login_bucket"] = pd.cut(
    first["days_since_last_app_login"],
    bins=[
        -np.inf,
        0,
        1,
        3,
        7,
        14,
        30,
        60,
        np.inf
    ],
    labels=[
        "0d",
        "1d",
        "2–3d",
        "4–7d",
        "8–14d",
        "15–30d",
        "31–60d",
        "60+d"
    ]
)


# ============================================================
# 5. DPD BUCKET
# ============================================================

first["dpd_bucket"] = pd.cut(
    first["days_past_due"],
    bins=[0, 3, 7, 15, 30],
    labels=[
        "01–03",
        "04–07",
        "08–15",
        "16–30"
    ],
    include_lowest=True
)


# ============================================================
# 6. APP LOGIN → PAYMENT
# UNCONDITIONAL VIEW
# ============================================================

login_summary = (
    first
    .groupby("login_bucket", observed=True)
    .agg(
        messages=("customer_id", "size"),
        payments=("payment_72h", "sum"),
        payment_rate=("payment_72h", "mean"),
        total_recovery=("recovery_72h", "sum"),
        recovery_per_message=("recovery_72h", "mean"),
        avg_dpd=("days_past_due", "mean"),
        median_dpd=("days_past_due", "median"),
        avg_balance=("outstanding_balance_brl", "mean")
    )
    .reset_index()
)

print("\n" + "=" * 100)
print("APP LOGIN RECENCY → PAYMENT")
print("FIRST CONTACT | DPD 1–30")
print("=" * 100)

display(
    login_summary.style.format({
        "payment_rate": "{:.2%}",
        "total_recovery": "R$ {:,.2f}",
        "recovery_per_message": "R$ {:,.2f}",
        "avg_dpd": "{:.1f}",
        "median_dpd": "{:.1f}",
        "avg_balance": "R$ {:,.2f}"
    })
)


# ============================================================
# 7. DPD → PAYMENT
# ============================================================

dpd_summary = (
    first
    .groupby("dpd_bucket", observed=True)
    .agg(
        messages=("customer_id", "size"),
        payments=("payment_72h", "sum"),
        payment_rate=("payment_72h", "mean"),
        total_recovery=("recovery_72h", "sum"),
        recovery_per_message=("recovery_72h", "mean"),
        avg_login_recency=("days_since_last_app_login", "mean"),
        median_login_recency=("days_since_last_app_login", "median")
    )
    .reset_index()
)

print("\n" + "=" * 100)
print("DPD → PAYMENT")
print("FIRST CONTACT")
print("=" * 100)

display(
    dpd_summary.style.format({
        "payment_rate": "{:.2%}",
        "total_recovery": "R$ {:,.2f}",
        "recovery_per_message": "R$ {:,.2f}",
        "avg_login_recency": "{:.1f}",
        "median_login_recency": "{:.1f}"
    })
)


# ============================================================
# 8. CRITICAL ANALYSIS:
# DPD × APP LOGIN
# ============================================================

cross = (
    first
    .groupby(
        ["dpd_bucket", "login_bucket"],
        observed=True
    )
    .agg(
        messages=("customer_id", "size"),
        payments=("payment_72h", "sum"),
        payment_rate=("payment_72h", "mean"),
        recovery=("recovery_72h", "sum"),
        recovery_per_message=("recovery_72h", "mean")
    )
    .reset_index()
)

print("\n" + "=" * 100)
print("DPD × APP LOGIN → PAYMENT")
print("=" * 100)

display(
    cross.style.format({
        "payment_rate": "{:.2%}",
        "recovery": "R$ {:,.2f}",
        "recovery_per_message": "R$ {:,.2f}"
    })
)


# ============================================================
# 9. HEATMAP TABLE — PAYMENT RATE
# ============================================================

payment_matrix = (
    cross.pivot(
        index="dpd_bucket",
        columns="login_bucket",
        values="payment_rate"
    )
)

print("\n" + "=" * 100)
print("PAYMENT RATE 72H")
print("=" * 100)

display(
    payment_matrix.style
    .format("{:.1%}")
    .background_gradient(axis=None)
)


# ============================================================
# 10. HEATMAP TABLE — RECOVERY / MESSAGE
# ============================================================

recovery_matrix = (
    cross.pivot(
        index="dpd_bucket",
        columns="login_bucket",
        values="recovery_per_message"
    )
)

print("\n" + "=" * 100)
print("RECOVERY / MESSAGE")
print("=" * 100)

display(
    recovery_matrix.style
    .format("R$ {:.2f}")
    .background_gradient(axis=None)
)


# ============================================================
# 11. HEATMAP TABLE — SAMPLE SIZE
# Essential to avoid interpreting tiny cells
# ============================================================

n_matrix = (
    cross.pivot(
        index="dpd_bucket",
        columns="login_bucket",
        values="messages"
    )
)

print("\n" + "=" * 100)
print("SAMPLE SIZE — N")
print("=" * 100)

display(
    n_matrix.style.format("{:,.0f}")
)


# ============================================================
# 12. DAILY DPD
#
# Don't impose buckets yet:
# inspect DPD 1, 2, 3...30
# ============================================================

daily = (
    first
    .groupby("days_past_due")
    .agg(
        messages=("customer_id", "size"),
        payments=("payment_72h", "sum"),
        payment_rate=("payment_72h", "mean"),
        recovery_per_message=("recovery_72h", "mean"),
        median_login_recency=("days_since_last_app_login", "median")
    )
    .reset_index()
)

print("\n" + "=" * 100)
print("DAILY DPD — FIRST CONTACT")
print("=" * 100)

display(
    daily.style.format({
        "payment_rate": "{:.2%}",
        "recovery_per_message": "R$ {:,.2f}",
        "median_login_recency": "{:.1f}"
    })
)

FIRST CONTACT — EARLY COLLECTIONS
Customers/messages : 11,723
Payment events     : 1,155
Payment rate       : 9.85%
Recovery           : R$ 762,592.41
Recovery / message : R$ 65.05
Missing app login  : 0

APP LOGIN RECENCY — DISTRIBUTION


,days_since_last_app_login
count,"11,723.00"
mean,19.03
std,20.02
min,0.00
1%,0.00
5%,1.00
10%,2.00
25%,5.00
50%,13.00
75%,26.00



APP LOGIN RECENCY → PAYMENT
FIRST CONTACT | DPD 1–30


,login_bucket,messages,payments,payment_rate,total_recovery,recovery_per_message,avg_dpd,median_dpd,avg_balance
0,0d,327,37,11.31%,"R$ 22,041.39",R$ 67.40,3.2,2.0,R$ 853.45
1,1d,647,63,9.74%,"R$ 35,474.16",R$ 54.83,3.1,2.0,R$ 844.05
2,2–3d,1092,115,10.53%,"R$ 66,508.85",R$ 60.91,3.0,2.0,R$ 832.52
3,4–7d,1891,201,10.63%,"R$ 139,730.35",R$ 73.89,3.0,2.0,R$ 833.39
4,8–14d,2419,255,10.54%,"R$ 163,797.90",R$ 67.71,3.0,2.0,R$ 856.57
5,15–30d,3020,271,8.97%,"R$ 188,478.05",R$ 62.41,3.1,2.0,R$ 861.89
6,31–60d,1793,179,9.98%,"R$ 124,952.25",R$ 69.69,3.0,2.0,R$ 854.08
7,60+d,534,34,6.37%,"R$ 21,609.46",R$ 40.47,3.2,3.0,R$ 839.24



DPD → PAYMENT
FIRST CONTACT


...

login_bucket,0d,1d,2–3d,4–7d,8–14d,15–30d,31–60d,60+d
dpd_bucket,,,,,,,,
01–03,R$ 66.06,R$ 56.94,R$ 56.76,R$ 78.56,R$ 68.31,R$ 63.60,R$ 70.72,R$ 39.51
04–07,R$ 47.09,R$ 41.38,R$ 68.84,R$ 71.21,R$ 66.63,R$ 60.23,R$ 57.21,R$ 39.17
08–15,R$ 142.83,R$ 95.41,R$ 66.61,R$ 28.39,R$ 66.21,R$ 58.37,R$ 118.34,R$ 57.16
16–30,R$ 180.88,R$ 0.00,R$ 281.80,R$ 62.75,R$ 0.00,R$ 68.90,R$ 0.00,R$ 0.00



SAMPLE SIZE — N


login_bucket,0d,1d,2–3d,4–7d,8–14d,15–30d,31–60d,60+d
dpd_bucket,,,,,,,,
01–03,221,430,759,"1,301","1,708","2,044","1,253",352
04–07,82,178,270,482,577,770,445,147
08–15,20,38,60,104,131,198,92,34
16–30,4,1,3,4,3,8,3,1



DAILY DPD — FIRST CONTACT


,days_past_due,messages,payments,payment_rate,recovery_per_message,median_login_recency
0,1,3841,386,10.05%,R$ 66.88,12.0
1,2,2482,245,9.87%,R$ 67.79,13.0
2,3,1745,170,9.74%,R$ 62.14,13.0
3,4,1241,128,10.31%,R$ 71.77,13.0
4,5,822,73,8.88%,R$ 57.10,13.0
5,6,558,48,8.60%,R$ 48.58,12.0
6,7,330,31,9.39%,R$ 51.71,13.0
7,8,228,23,10.09%,R$ 67.34,13.0
8,9,162,15,9.26%,R$ 63.26,15.0
9,10,101,12,11.88%,R$ 65.53,11.0


In [7]:
# ============================================================
# SHARE OF TOTAL RECOVERY BY DPD
# FIRST CONTACT | DPD 1–30
# ============================================================

import numpy as np
import pandas as pd

# first já foi construído anteriormente:
# 1 linha = primeiro contato observado por cliente
# DPD 1–30
# recovery_72h = amount_paid_brl quando paid_within_72h == True

recovery_by_dpd = (
    first
    .groupby("days_past_due", as_index=False)
    .agg(
        customers=("customer_id", "size"),
        payers=("payment_72h", "sum"),
        recovered_brl=("recovery_72h", "sum"),
        recovery_per_message=("recovery_72h", "mean"),
        payment_rate=("payment_72h", "mean")
    )
)

# ------------------------------------------------------------
# Total recovery
# ------------------------------------------------------------

total_recovery = recovery_by_dpd["recovered_brl"].sum()

# ------------------------------------------------------------
# Share of ALL recovered money
# ------------------------------------------------------------

recovery_by_dpd["recovery_share"] = (
    recovery_by_dpd["recovered_brl"] / total_recovery
)

# cumulative share
recovery_by_dpd["cumulative_recovery_share"] = (
    recovery_by_dpd["recovery_share"].cumsum()
)

# customer volume share — useful comparison
total_customers = recovery_by_dpd["customers"].sum()

recovery_by_dpd["customer_share"] = (
    recovery_by_dpd["customers"] / total_customers
)

# Difference:
# Is this DPD concentrating more/less recovery than volume?
recovery_by_dpd["recovery_vs_volume_pp"] = (
    recovery_by_dpd["recovery_share"]
    - recovery_by_dpd["customer_share"]
)

print("=" * 110)
print("SHARE OF TOTAL RECOVERY BY DPD — FIRST CONTACT")
print("=" * 110)

print(f"Total customers : {total_customers:,}")
print(f"Total recovery  : R$ {total_recovery:,.2f}")

display(
    recovery_by_dpd.style.format({
        "customers": "{:,.0f}",
        "payers": "{:,.0f}",
        "recovered_brl": "R$ {:,.2f}",
        "recovery_per_message": "R$ {:,.2f}",
        "payment_rate": "{:.2%}",
        "recovery_share": "{:.2%}",
        "cumulative_recovery_share": "{:.2%}",
        "customer_share": "{:.2%}",
        "recovery_vs_volume_pp": "{:+.2%}"
    })
)

SHARE OF TOTAL RECOVERY BY DPD — FIRST CONTACT
Total customers : 11,723
Total recovery  : R$ 762,592.41


,days_past_due,customers,payers,recovered_brl,recovery_per_message,payment_rate,recovery_share,cumulative_recovery_share,customer_share,recovery_vs_volume_pp
0,1,"3,841",386,"R$ 256,870.70",R$ 66.88,10.05%,33.68%,33.68%,32.76%,+0.92%
1,2,"2,482",245,"R$ 168,244.26",R$ 67.79,9.87%,22.06%,55.75%,21.17%,+0.89%
2,3,"1,745",170,"R$ 108,441.01",R$ 62.14,9.74%,14.22%,69.97%,14.89%,-0.67%
3,4,"1,241",128,"R$ 89,063.03",R$ 71.77,10.31%,11.68%,81.65%,10.59%,+1.09%
4,5,822,73,"R$ 46,936.31",R$ 57.10,8.88%,6.15%,87.80%,7.01%,-0.86%
5,6,558,48,"R$ 27,109.58",R$ 48.58,8.60%,3.55%,91.35%,4.76%,-1.20%
6,7,330,31,"R$ 17,063.32",R$ 51.71,9.39%,2.24%,93.59%,2.81%,-0.58%
7,8,228,23,"R$ 15,352.95",R$ 67.34,10.09%,2.01%,95.61%,1.94%,+0.07%
8,9,162,15,"R$ 10,247.51",R$ 63.26,9.26%,1.34%,96.95%,1.38%,-0.04%
9,10,101,12,"R$ 6,618.68",R$ 65.53,11.88%,0.87%,97.82%,0.86%,+0.01%


In [8]:
# ============================================================
# SHARE OF TOTAL RECOVERY BY DPD
# ALL HISTORICAL RECOVERY
# ============================================================

import numpy as np
import pandas as pd

x = wa.copy()

x["days_past_due"] = pd.to_numeric(
    x["days_past_due"],
    errors="coerce"
)

x["amount_paid_brl"] = pd.to_numeric(
    x["amount_paid_brl"],
    errors="coerce"
).fillna(0)

x["payment_72h"] = (
    x["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

# Recovery attributed to the message
x["recovery_72h"] = np.where(
    x["payment_72h"],
    x["amount_paid_brl"],
    0
)

# ============================================================
# DPD 1–30
# ============================================================

early = x.loc[
    x["days_past_due"].between(1, 30)
].copy()

# ============================================================
# RECOVERY BY DPD
# ============================================================

recovery_dpd = (
    early
    .groupby("days_past_due", as_index=False)
    .agg(
        messages=("message_id", "size"),
        customers=("customer_id", "nunique"),
        payment_events=("payment_72h", "sum"),
        recovered_brl=("recovery_72h", "sum")
    )
)

# ============================================================
# IMPORTANT:
# denominator = ALL recovery in the entire WA history
# ============================================================

total_recovery_all = x["recovery_72h"].sum()

recovery_dpd["share_total_recovery"] = (
    recovery_dpd["recovered_brl"]
    / total_recovery_all
)

recovery_dpd["cumulative_share_total"] = (
    recovery_dpd["share_total_recovery"].cumsum()
)

# Also useful:
# share only within DPD 1–30
total_recovery_early = early["recovery_72h"].sum()

recovery_dpd["share_early_recovery"] = (
    recovery_dpd["recovered_brl"]
    / total_recovery_early
)

recovery_dpd["cumulative_share_early"] = (
    recovery_dpd["share_early_recovery"].cumsum()
)

print("=" * 110)
print("RECOVERY SHARE BY DPD — ALL HISTORICAL PAYMENTS")
print("=" * 110)

print(f"TOTAL WA RECOVERY      : R$ {total_recovery_all:,.2f}")
print(f"RECOVERY DPD 1–30      : R$ {total_recovery_early:,.2f}")
print(
    f"% recovered by DPD 30 : "
    f"{total_recovery_early / total_recovery_all:.2%}"
)

display(
    recovery_dpd.style.format({
        "messages": "{:,.0f}",
        "customers": "{:,.0f}",
        "payment_events": "{:,.0f}",
        "recovered_brl": "R$ {:,.2f}",
        "share_total_recovery": "{:.2%}",
        "cumulative_share_total": "{:.2%}",
        "share_early_recovery": "{:.2%}",
        "cumulative_share_early": "{:.2%}"
    })
)

RECOVERY SHARE BY DPD — ALL HISTORICAL PAYMENTS
TOTAL WA RECOVERY      : R$ 3,459,305.30
RECOVERY DPD 1–30      : R$ 3,028,175.92
% recovered by DPD 30 : 87.54%


,days_past_due,messages,customers,payment_events,recovered_brl,share_total_recovery,cumulative_share_total,share_early_recovery,cumulative_share_early
0,1,"3,841","3,841",386,"R$ 256,870.70",7.43%,7.43%,8.48%,8.48%
1,2,"3,632","3,632",346,"R$ 233,301.69",6.74%,14.17%,7.70%,16.19%
2,3,"3,427","3,427",330,"R$ 205,822.52",5.95%,20.12%,6.80%,22.98%
3,4,"3,341","3,341",298,"R$ 205,215.71",5.93%,26.05%,6.78%,29.76%
4,5,"3,276","3,276",275,"R$ 183,204.59",5.30%,31.35%,6.05%,35.81%
5,6,"3,115","3,115",279,"R$ 167,880.37",4.85%,36.20%,5.54%,41.35%
6,7,"3,092","3,092",236,"R$ 148,801.59",4.30%,40.50%,4.91%,46.27%
7,8,"3,063","3,063",265,"R$ 169,760.96",4.91%,45.41%,5.61%,51.87%
8,9,"2,889","2,889",247,"R$ 147,154.55",4.25%,49.66%,4.86%,56.73%
9,10,"2,809","2,809",218,"R$ 130,811.38",3.78%,53.44%,4.32%,61.05%


In [10]:
# ============================================================
# FIRST MESSAGE — WHICH DPD 1–7 PERFORMS BETTER?
# Fast statistical analysis
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import kruskal, mannwhitneyu
from statsmodels.stats.multitest import multipletests

# ============================================================
# 1. REBUILD FIRST MESSAGE PER CUSTOMER
# ============================================================

tmp = wa.copy()

tmp["sent_at"] = pd.to_datetime(tmp["sent_at"])

tmp["days_past_due"] = pd.to_numeric(
    tmp["days_past_due"],
    errors="coerce"
)

tmp["amount_paid_brl"] = pd.to_numeric(
    tmp["amount_paid_brl"],
    errors="coerce"
).fillna(0)

tmp["paid_within_72h"] = (
    tmp["paid_within_72h"]
    .fillna(False)
    .astype(bool)
)

tmp["recovery_72h"] = np.where(
    tmp["paid_within_72h"],
    tmp["amount_paid_brl"],
    0
)

# first observed WhatsApp
first_msg = (
    tmp
    .sort_values(["customer_id", "sent_at", "message_id"])
    .groupby("customer_id", as_index=False)
    .first()
)

# DPD 1–7 only
d = first_msg.loc[
    first_msg["days_past_due"].between(1, 7)
].copy()

d["days_past_due"] = d["days_past_due"].astype(int)


# ============================================================
# 2. DESCRIPTIVE RESULTS
# ============================================================

summary = (
    d.groupby("days_past_due")
    .agg(
        N=("customer_id", "size"),
        payers=("paid_within_72h", "sum"),
        payment_rate=("paid_within_72h", "mean"),
        recovery=("recovery_72h", "sum"),
        recovery_per_message=("recovery_72h", "mean")
    )
    .reset_index()
)

print("=" * 90)
print("FIRST MESSAGE — DPD 1–7")
print("=" * 90)

print(summary.to_string(
    index=False,
    formatters={
        "payment_rate": lambda x: f"{x:.2%}",
        "recovery": lambda x: f"R$ {x:,.2f}",
        "recovery_per_message": lambda x: f"R$ {x:,.2f}"
    }
))


# ============================================================
# 3. GLOBAL TEST
# H0 = recovery distribution is the same across DPD 1–7
# ============================================================

groups = [
    d.loc[
        d["days_past_due"].eq(day),
        "recovery_72h"
    ].values
    for day in range(1, 8)
]

stat, p_global = kruskal(*groups)

print("\n" + "=" * 90)
print("GLOBAL TEST — DPD 1–7")
print("=" * 90)

print(f"Kruskal-Wallis statistic : {stat:.4f}")
print(f"p-value                  : {p_global:.6f}")

if p_global < 0.05:
    print("RESULT: There is evidence of differences between at least two DPDs.")
else:
    print("RESULT: No evidence of differences across DPD 1–7.")


# ============================================================
# 4. EACH DAY AGAINST EVERY OTHER DAY
#
# Example:
# DPD 1 versus customers contacted on DPD 2–7
# DPD 2 versus customers contacted on DPD 1,3–7
# ...
# ============================================================

rows = []

for day in range(1, 8):

    A = d.loc[
        d["days_past_due"].eq(day),
        "recovery_72h"
    ]

    B = d.loc[
        d["days_past_due"].ne(day),
        "recovery_72h"
    ]

    stat, p = mannwhitneyu(
        A,
        B,
        alternative="two-sided"
    )

    rows.append({
        "DPD": day,
        "N_day": len(A),
        "N_rest": len(B),
        "RPM_day": A.mean(),
        "RPM_rest": B.mean(),
        "difference": A.mean() - B.mean(),
        "uplift_vs_rest": (
            A.mean() / B.mean() - 1
        ),
        "p_raw": p
    })

day_vs_rest = pd.DataFrame(rows)

# Holm correction for 7 tests
day_vs_rest["p_holm"] = multipletests(
    day_vs_rest["p_raw"],
    method="holm"
)[1]

day_vs_rest["significant"] = (
    day_vs_rest["p_holm"] < 0.05
)

print("\n" + "=" * 90)
print("EACH DPD VS ALL OTHER DPDs")
print("=" * 90)

print(
    day_vs_rest.to_string(
        index=False,
        formatters={
            "RPM_day": lambda x: f"R$ {x:,.2f}",
            "RPM_rest": lambda x: f"R$ {x:,.2f}",
            "difference": lambda x: f"R$ {x:+,.2f}",
            "uplift_vs_rest": lambda x: f"{x:+.1%}",
            "p_raw": lambda x: f"{x:.6f}",
            "p_holm": lambda x: f"{x:.6f}"
        }
    )
)

FIRST MESSAGE — DPD 1–7
 days_past_due    N  payers payment_rate      recovery recovery_per_message
             1 3841     386       10.05% R$ 256,870.70             R$ 66.88
             2 2482     245        9.87% R$ 168,244.26             R$ 67.79
             3 1745     170        9.74% R$ 108,441.01             R$ 62.14
             4 1241     128       10.31%  R$ 89,063.03             R$ 71.77
             5  822      73        8.88%  R$ 46,936.31             R$ 57.10
             6  558      48        8.60%  R$ 27,109.58             R$ 48.58
             7  330      31        9.39%  R$ 17,063.32             R$ 51.71

GLOBAL TEST — DPD 1–7
Kruskal-Wallis statistic : 2.8207
p-value                  : 0.830994
RESULT: No evidence of differences across DPD 1–7.

EACH DPD VS ALL OTHER DPDs
 DPD  N_day  N_rest  RPM_day RPM_rest difference uplift_vs_rest    p_raw   p_holm  significant
   1   3841    7178 R$ 66.88 R$ 63.65   R$ +3.23          +5.1% 0.508864 1.000000        False
   2  

In [11]:
# ============================================================
# FIRST MESSAGE
# DPD 1–7 × APP LOGIN RECENCY
#
# Goal:
# Does the best timing for message #1 change according
# to customer's app-login recency?
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. BASE
# first_msg already = first observed WA message per customer
# ------------------------------------------------------------

d = first_msg.loc[
    first_msg["days_past_due"].between(1, 7)
].copy()

d["days_past_due"] = d["days_past_due"].astype(int)

d["days_since_last_app_login"] = pd.to_numeric(
    d["days_since_last_app_login"],
    errors="coerce"
)

# ------------------------------------------------------------
# 2. LOGIN BUCKET
# ------------------------------------------------------------

d["login_bucket"] = pd.cut(
    d["days_since_last_app_login"],
    bins=[
        -np.inf,
        0,
        1,
        3,
        7,
        14,
        30,
        60,
        np.inf
    ],
    labels=[
        "0d",
        "1d",
        "2–3d",
        "4–7d",
        "8–14d",
        "15–30d",
        "31–60d",
        "60+d"
    ]
)

# ============================================================
# 3. DPD × LOGIN
# ============================================================

cross = (
    d.groupby(
        ["login_bucket", "days_past_due"],
        observed=True
    )
    .agg(
        N=("customer_id", "size"),
        payers=("paid_within_72h", "sum"),
        payment_rate=("paid_within_72h", "mean"),
        recovered_brl=("recovery_72h", "sum"),
        recovery_per_message=("recovery_72h", "mean")
    )
    .reset_index()
)

# ============================================================
# 4. N MATRIX
# ============================================================

n_matrix = cross.pivot(
    index="login_bucket",
    columns="days_past_due",
    values="N"
)

print("=" * 100)
print("N — APP LOGIN × DPD OF FIRST MESSAGE")
print("=" * 100)

display(
    n_matrix.style.format("{:,.0f}")
)

# ============================================================
# 5. PAYMENT RATE MATRIX
# ============================================================

payment_matrix = cross.pivot(
    index="login_bucket",
    columns="days_past_due",
    values="payment_rate"
)

print("\n" + "=" * 100)
print("PAYMENT RATE 72H — APP LOGIN × DPD")
print("=" * 100)

display(
    payment_matrix.style
    .format("{:.1%}")
    .background_gradient(axis=None)
)

# ============================================================
# 6. RECOVERY / MESSAGE MATRIX
# ============================================================

rpm_matrix = cross.pivot(
    index="login_bucket",
    columns="days_past_due",
    values="recovery_per_message"
)

print("\n" + "=" * 100)
print("RECOVERY / MESSAGE — APP LOGIN × DPD")
print("=" * 100)

display(
    rpm_matrix.style
    .format("R$ {:,.2f}")
    .background_gradient(axis=None)
)

# ============================================================
# 7. BEST OBSERVED DPD WITHIN EACH LOGIN PROFILE
# descriptive only
# ============================================================

best_observed = (
    cross
    .sort_values(
        ["login_bucket", "recovery_per_message"],
        ascending=[True, False]
    )
    .groupby("login_bucket", observed=True)
    .first()
    .reset_index()
    [
        [
            "login_bucket",
            "days_past_due",
            "N",
            "payers",
            "payment_rate",
            "recovery_per_message"
        ]
    ]
)

print("\n" + "=" * 100)
print("BEST OBSERVED DPD WITHIN EACH APP-LOGIN PROFILE")
print("DESCRIPTIVE ONLY — NOT YET STATISTICALLY VALIDATED")
print("=" * 100)

display(
    best_observed.style.format({
        "N": "{:,.0f}",
        "payers": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery_per_message": "R$ {:,.2f}"
    })
)

N — APP LOGIN × DPD OF FIRST MESSAGE


days_past_due,1,2,3,4,5,6,7
login_bucket,,,,,,,
0d,111,69,41,29,16,30,7
1d,193,142,95,79,42,34,23
2–3d,367,225,167,114,79,44,33
4–7d,638,390,273,198,134,93,57
8–14d,817,525,366,235,173,106,63
15–30d,963,615,466,326,211,139,94
31–60d,596,409,248,196,125,85,39
60+d,156,107,89,64,42,27,14



PAYMENT RATE 72H — APP LOGIN × DPD


days_past_due,1,2,3,4,5,6,7
login_bucket,,,,,,,
0d,11.7%,5.8%,19.5%,10.3%,6.2%,6.7%,14.3%
1d,7.3%,11.3%,10.5%,8.9%,11.9%,8.8%,8.7%
2–3d,11.7%,8.9%,7.8%,11.4%,17.7%,2.3%,6.1%
4–7d,10.8%,12.1%,11.4%,13.6%,4.5%,7.5%,8.8%
8–14d,10.4%,9.5%,10.7%,11.5%,8.7%,16.0%,9.5%
15–30d,9.0%,9.4%,10.7%,8.0%,8.5%,5.8%,10.6%
31–60d,11.1%,10.3%,5.6%,10.7%,8.8%,10.6%,7.7%
60+d,5.8%,7.5%,5.6%,6.2%,7.1%,3.7%,14.3%



RECOVERY / MESSAGE — APP LOGIN × DPD


days_past_due,1,2,3,4,5,6,7
login_bucket,,,,,,,
0d,R$ 82.30,R$ 23.23,R$ 94.18,R$ 66.69,R$ 33.41,R$ 27.15,R$ 82.53
1d,R$ 44.49,R$ 64.58,R$ 70.81,R$ 28.01,R$ 44.05,R$ 29.71,R$ 99.67
2–3d,R$ 56.17,R$ 59.70,R$ 54.08,R$ 65.43,R$ 118.56,R$ 4.96,R$ 46.78
4–7d,R$ 77.04,R$ 87.72,R$ 69.02,R$ 103.67,R$ 49.29,R$ 47.62,R$ 48.48
8–14d,R$ 67.84,R$ 66.43,R$ 72.08,R$ 77.69,R$ 54.24,R$ 74.78,R$ 45.67
15–30d,R$ 62.36,R$ 62.18,R$ 68.03,R$ 65.29,R$ 60.54,R$ 49.51,R$ 57.81
31–60d,R$ 80.74,R$ 78.75,R$ 33.36,R$ 78.06,R$ 41.01,R$ 46.81,R$ 27.01
60+d,R$ 37.09,R$ 42.08,R$ 40.68,R$ 32.63,R$ 30.85,R$ 68.60,R$ 37.21



BEST OBSERVED DPD WITHIN EACH APP-LOGIN PROFILE
DESCRIPTIVE ONLY — NOT YET STATISTICALLY VALIDATED


,login_bucket,days_past_due,N,payers,payment_rate,recovery_per_message
0,0d,3,41,8,19.51%,R$ 94.18
1,1d,7,23,2,8.70%,R$ 99.67
2,2–3d,5,79,14,17.72%,R$ 118.56
3,4–7d,4,198,27,13.64%,R$ 103.67
4,8–14d,4,235,27,11.49%,R$ 77.69
5,15–30d,3,466,50,10.73%,R$ 68.03
6,31–60d,1,596,66,11.07%,R$ 80.74
7,60+d,6,27,1,3.70%,R$ 68.60


In [12]:
# ============================================================
# APP LOGIN PROFILE × DPD 1–7
# FIRST MESSAGE ONLY
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import kruskal

z = first_msg.loc[
    first_msg["days_past_due"].between(1, 7)
].copy()

z["days_past_due"] = z["days_past_due"].astype(int)

z["days_since_last_app_login"] = pd.to_numeric(
    z["days_since_last_app_login"],
    errors="coerce"
)

# ------------------------------------------------------------
# Consolidated login profiles
# ------------------------------------------------------------

z["login_profile"] = pd.cut(
    z["days_since_last_app_login"],
    bins=[-np.inf, 3, 14, 30, np.inf],
    labels=[
        "≤3d",
        "4–14d",
        "15–30d",
        ">30d"
    ]
)

# ------------------------------------------------------------
# Descriptive matrix
# ------------------------------------------------------------

summary = (
    z.groupby(
        ["login_profile", "days_past_due"],
        observed=True
    )
    .agg(
        N=("customer_id", "size"),
        payers=("paid_within_72h", "sum"),
        payment_rate=("paid_within_72h", "mean"),
        recovery_per_message=("recovery_72h", "mean")
    )
    .reset_index()
)

print("=" * 100)
print("N")
print("=" * 100)

display(
    summary.pivot(
        index="login_profile",
        columns="days_past_due",
        values="N"
    ).style.format("{:,.0f}")
)

print("\n" + "=" * 100)
print("PAYMENT RATE")
print("=" * 100)

display(
    summary.pivot(
        index="login_profile",
        columns="days_past_due",
        values="payment_rate"
    ).style.format("{:.1%}")
)

print("\n" + "=" * 100)
print("RECOVERY / MESSAGE")
print("=" * 100)

display(
    summary.pivot(
        index="login_profile",
        columns="days_past_due",
        values="recovery_per_message"
    ).style.format("R$ {:,.2f}")
)

# ------------------------------------------------------------
# Global test inside each login profile
# ------------------------------------------------------------

tests = []

for profile in z["login_profile"].dropna().unique():

    temp = z.loc[
        z["login_profile"].eq(profile)
    ]

    groups = [
        temp.loc[
            temp["days_past_due"].eq(day),
            "recovery_72h"
        ].values
        for day in range(1, 8)
    ]

    groups = [g for g in groups if len(g) > 0]

    stat, p = kruskal(*groups)

    tests.append({
        "login_profile": profile,
        "N": len(temp),
        "kruskal_stat": stat,
        "p_value": p
    })

tests = pd.DataFrame(tests)

print("\n" + "=" * 100)
print("DOES DPD MATTER WITHIN EACH APP-LOGIN PROFILE?")
print("=" * 100)

print(
    tests.to_string(
        index=False,
        formatters={
            "kruskal_stat": lambda x: f"{x:.4f}",
            "p_value": lambda x: f"{x:.6f}"
        }
    )
)

N


days_past_due,1,2,3,4,5,6,7
login_profile,,,,,,,
≤3d,671,436,303,222,137,108,63
4–14d,"1,455",915,639,433,307,199,120
15–30d,963,615,466,326,211,139,94
>30d,752,516,337,260,167,112,53



PAYMENT RATE


days_past_due,1,2,3,4,5,6,7
login_profile,,,,,,,
≤3d,10.4%,9.2%,10.2%,10.4%,14.6%,5.6%,7.9%
4–14d,10.6%,10.6%,11.0%,12.5%,6.8%,12.1%,9.2%
15–30d,9.0%,9.4%,10.7%,8.0%,8.5%,5.8%,10.6%
>30d,10.0%,9.7%,5.6%,9.6%,8.4%,8.9%,9.4%



RECOVERY / MESSAGE


days_past_due,1,2,3,4,5,6,7
login_profile,,,,,,,
≤3d,R$ 57.13,R$ 55.52,R$ 64.75,R$ 52.28,R$ 85.77,R$ 18.91,R$ 70.06
4–14d,R$ 71.87,R$ 75.50,R$ 70.77,R$ 89.57,R$ 52.08,R$ 62.08,R$ 47.00
15–30d,R$ 62.36,R$ 62.18,R$ 68.03,R$ 65.29,R$ 60.54,R$ 49.51,R$ 57.81
>30d,R$ 71.69,R$ 71.15,R$ 35.29,R$ 66.88,R$ 38.46,R$ 52.06,R$ 29.70



DOES DPD MATTER WITHIN EACH APP-LOGIN PROFILE?
login_profile    N kruskal_stat  p_value
         >30d 2197       6.2457 0.396240
       15–30d 2814       3.8607 0.695520
        4–14d 4068       6.8496 0.334979
          ≤3d 1940       6.4208 0.377739


In [13]:
# ============================================================
# HETEROGENEITY SCREENING
# FIRST MESSAGE — DPD 1–7
# ONLY VARIABLES AVAILABLE IN BOTH WA + QUEUE
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests

# ============================================================
# 1. BASE — FIRST MESSAGE ONLY
# ============================================================

z = first_msg.loc[
    first_msg["days_past_due"].between(1, 7)
].copy()

z["days_past_due"] = pd.to_numeric(
    z["days_past_due"], errors="coerce"
).astype(int)

# ------------------------------------------------------------
# Numeric variables
# ------------------------------------------------------------

numeric_vars = [
    "outstanding_balance_brl",
    "monthly_salary_brl",
    "n_prior_transactions",
    "account_age_months",
    "days_since_last_app_login"
]

for col in numeric_vars:
    z[col] = pd.to_numeric(z[col], errors="coerce")


# ============================================================
# 2. DERIVED VARIABLE — BALANCE / SALARY
# ============================================================

z["balance_to_salary"] = np.where(
    z["monthly_salary_brl"] > 0,
    z["outstanding_balance_brl"] /
    z["monthly_salary_brl"],
    np.nan
)

numeric_vars.append("balance_to_salary")


# ============================================================
# 3. CREATE QUARTILES
# ============================================================

segment_vars = []

for col in numeric_vars:

    new_col = f"{col}_Q"

    try:

        z[new_col] = pd.qcut(
            z[col],
            q=4,
            labels=["Q1", "Q2", "Q3", "Q4"],
            duplicates="drop"
        )

        segment_vars.append(new_col)

    except Exception as e:
        print(f"Could not create quartiles for {col}: {e}")


# ============================================================
# 4. STATE
# Only states with reasonable volume
# ============================================================

state_counts = z["state_uf"].value_counts()

valid_states = state_counts[
    state_counts >= 300
].index

z["state_segment"] = np.where(
    z["state_uf"].isin(valid_states),
    z["state_uf"],
    "OTHER"
)

segment_vars.append("state_segment")


# ============================================================
# 5. PAYDAY
# Use broader buckets to preserve N
# ============================================================

z["payday_day_of_month"] = pd.to_numeric(
    z["payday_day_of_month"],
    errors="coerce"
)

z["payday_segment"] = pd.cut(
    z["payday_day_of_month"],
    bins=[0, 5, 10, 15, 20, 25, 31],
    labels=[
        "01–05",
        "06–10",
        "11–15",
        "16–20",
        "21–25",
        "26–31"
    ],
    include_lowest=True
)

segment_vars.append("payday_segment")


# ============================================================
# 6. SCREENING
#
# For every variable × segment:
# test whether recovery differs across DPD 1–7
#
# Minimum:
# 30 observations per DPD cell
# ============================================================

results = []

MIN_CELL = 30

for variable in segment_vars:

    for segment in z[variable].dropna().unique():

        temp = z.loc[
            z[variable].eq(segment)
        ].copy()

        counts = (
            temp["days_past_due"]
            .value_counts()
            .reindex(range(1, 8), fill_value=0)
        )

        # Require reasonable support in ALL 7 DPDs
        valid_support = (counts >= MIN_CELL).all()

        if not valid_support:
            continue

        groups = [
            temp.loc[
                temp["days_past_due"].eq(day),
                "recovery_72h"
            ].dropna().values
            for day in range(1, 8)
        ]

        stat, p = kruskal(*groups)

        # --------------------------------------------
        # Best observed DPD
        # --------------------------------------------

        daily = (
            temp.groupby("days_past_due")
            .agg(
                N=("customer_id", "size"),
                payment_rate=("paid_within_72h", "mean"),
                RPM=("recovery_72h", "mean")
            )
        )

        best_day = daily["RPM"].idxmax()
        best_rpm = daily.loc[best_day, "RPM"]

        # compare with DPD1
        dpd1_rpm = daily.loc[1, "RPM"]

        uplift_vs_dpd1 = (
            best_rpm / dpd1_rpm - 1
            if dpd1_rpm != 0
            else np.nan
        )

        results.append({
            "variable": variable,
            "segment": str(segment),

            "N_total": len(temp),
            "min_N_cell": counts.min(),

            "best_DPD": best_day,
            "best_RPM": best_rpm,

            "DPD1_RPM": dpd1_rpm,

            "best_vs_DPD1":
                best_rpm - dpd1_rpm,

            "uplift_vs_DPD1":
                uplift_vs_dpd1,

            "kruskal_stat": stat,
            "p_raw": p
        })


screen = pd.DataFrame(results)


# ============================================================
# 7. MULTIPLE TESTING CORRECTION
# ============================================================

if len(screen) > 0:

    screen["p_holm"] = multipletests(
        screen["p_raw"],
        method="holm"
    )[1]

    screen["significant_raw"] = (
        screen["p_raw"] < 0.05
    )

    screen["significant_holm"] = (
        screen["p_holm"] < 0.05
    )

    screen = screen.sort_values(
        ["p_holm", "p_raw"]
    ).reset_index(drop=True)


# ============================================================
# 8. OUTPUT
# ============================================================

print("=" * 120)
print("SCREENING — DOES FIRST-CONTACT TIMING DIFFER BY CUSTOMER PROFILE?")
print("=" * 120)

print(f"Population DPD1–7 : {len(z):,}")
print(f"Segments tested   : {len(screen):,}")
print(f"Minimum N/cell    : {MIN_CELL}")

display(
    screen.style.format({
        "best_RPM": "R$ {:,.2f}",
        "DPD1_RPM": "R$ {:,.2f}",
        "best_vs_DPD1": "R$ {:+,.2f}",
        "uplift_vs_DPD1": "{:+.1%}",
        "kruskal_stat": "{:.3f}",
        "p_raw": "{:.4f}",
        "p_holm": "{:.4f}"
    })
)


# ============================================================
# 9. ONLY POTENTIAL SIGNALS
# ============================================================

signals = screen.loc[
    (screen["p_raw"] < 0.10)
].copy()

print("\n" + "=" * 120)
print("POTENTIAL SIGNALS — p_raw < 0.10")
print("EXPLORATORY ONLY")
print("=" * 120)

if len(signals):

    display(
        signals.style.format({
            "best_RPM": "R$ {:,.2f}",
            "DPD1_RPM": "R$ {:,.2f}",
            "best_vs_DPD1": "R$ {:+,.2f}",
            "uplift_vs_DPD1": "{:+.1%}",
            "p_raw": "{:.4f}",
            "p_holm": "{:.4f}"
        })
    )

else:
    print("No potential timing heterogeneity found.")

SCREENING — DOES FIRST-CONTACT TIMING DIFFER BY CUSTOMER PROFILE?
Population DPD1–7 : 11,019
Segments tested   : 30
Minimum N/cell    : 30


,variable,segment,N_total,min_N_cell,best_DPD,best_RPM,DPD1_RPM,best_vs_DPD1,uplift_vs_DPD1,kruskal_stat,p_raw,p_holm,significant_raw,significant_holm
0,payday_segment,16–20,1658,54,2,R$ 102.30,R$ 60.51,R$ +41.79,+69.1%,15.038,0.0200,0.5989,True,False
1,monthly_salary_brl_Q,Q2,2746,81,4,R$ 59.61,R$ 50.31,R$ +9.30,+18.5%,10.151,0.1184,1.0000,False,False
2,balance_to_salary_Q,Q4,2755,78,1,R$ 91.91,R$ 91.91,R$ +0.00,+0.0%,9.598,0.1426,1.0000,False,False
3,balance_to_salary_Q,Q3,2754,85,4,R$ 102.81,R$ 66.49,R$ +36.32,+54.6%,9.454,0.1496,1.0000,False,False
4,state_segment,SP,2429,76,3,R$ 90.56,R$ 66.25,R$ +24.30,+36.7%,7.526,0.2749,1.0000,False,False
5,n_prior_transactions_Q,Q2,2410,59,4,R$ 86.38,R$ 68.05,R$ +18.33,+26.9%,6.388,0.3812,1.0000,False,False
6,days_since_last_app_login_Q,Q4,2678,61,1,R$ 68.65,R$ 68.65,R$ +0.00,+0.0%,6.290,0.3915,1.0000,False,False
7,outstanding_balance_brl_Q,Q3,2754,81,4,R$ 80.21,R$ 64.37,R$ +15.84,+24.6%,5.484,0.4834,1.0000,False,False
8,outstanding_balance_brl_Q,Q4,2755,88,2,R$ 125.40,R$ 122.69,R$ +2.71,+2.2%,5.433,0.4896,1.0000,False,False
9,monthly_salary_brl_Q,Q4,2735,84,4,R$ 120.40,R$ 106.81,R$ +13.59,+12.7%,5.246,0.5127,1.0000,False,False



POTENTIAL SIGNALS — p_raw < 0.10
EXPLORATORY ONLY


,variable,segment,N_total,min_N_cell,best_DPD,best_RPM,DPD1_RPM,best_vs_DPD1,uplift_vs_DPD1,kruskal_stat,p_raw,p_holm,significant_raw,significant_holm
0,payday_segment,16–20,1658,54,2,R$ 102.30,R$ 60.51,R$ +41.79,+69.1%,15.037891,0.0200,0.5989,True,False


In [14]:
# ============================================================
# DEEP DIVE
# EXACT PAYDAY 16–20 × DPD 1–7
# FIRST MESSAGE ONLY
# ============================================================

import numpy as np
import pandas as pd

x = first_msg.loc[
    first_msg["days_past_due"].between(1, 7)
].copy()

x["days_past_due"] = pd.to_numeric(
    x["days_past_due"], errors="coerce"
).astype(int)

x["payday_day_of_month"] = pd.to_numeric(
    x["payday_day_of_month"], errors="coerce"
)

# only payday 16–20
x = x.loc[
    x["payday_day_of_month"].between(16, 20)
].copy()

# ============================================================
# SUMMARY
# ============================================================

cross = (
    x.groupby(
        ["payday_day_of_month", "days_past_due"]
    )
    .agg(
        N=("customer_id", "size"),
        payers=("paid_within_72h", "sum"),
        payment_rate=("paid_within_72h", "mean"),
        recovery_brl=("recovery_72h", "sum"),
        RPM=("recovery_72h", "mean")
    )
    .reset_index()
)

# ============================================================
# 1. N
# ============================================================

N = cross.pivot(
    index="payday_day_of_month",
    columns="days_past_due",
    values="N"
)

print("=" * 100)
print("N — EXACT PAYDAY × DPD FIRST MESSAGE")
print("=" * 100)

display(
    N.style
    .format("{:,.0f}")
    .background_gradient(axis=None)
)

# ============================================================
# 2. PAYMENT RATE
# ============================================================

PAY = cross.pivot(
    index="payday_day_of_month",
    columns="days_past_due",
    values="payment_rate"
)

print("\n" + "=" * 100)
print("PAYMENT RATE 72H — EXACT PAYDAY × DPD")
print("=" * 100)

display(
    PAY.style
    .format("{:.1%}")
    .background_gradient(axis=None)
)

# ============================================================
# 3. RECOVERY / MESSAGE
# ============================================================

RPM = cross.pivot(
    index="payday_day_of_month",
    columns="days_past_due",
    values="RPM"
)

print("\n" + "=" * 100)
print("RECOVERY / MESSAGE — EXACT PAYDAY × DPD")
print("=" * 100)

display(
    RPM.style
    .format("R$ {:,.2f}")
    .background_gradient(axis=None)
)

# ============================================================
# 4. DPD2 VS DPD1 BY EXACT PAYDAY
# ============================================================

comparison = []

for payday in range(16, 21):

    t = x.loc[
        x["payday_day_of_month"].eq(payday)
    ]

    d1 = t.loc[
        t["days_past_due"].eq(1)
    ]

    d2 = t.loc[
        t["days_past_due"].eq(2)
    ]

    rpm1 = d1["recovery_72h"].mean()
    rpm2 = d2["recovery_72h"].mean()

    comparison.append({
        "payday": payday,

        "N_DPD1": len(d1),
        "N_DPD2": len(d2),

        "pay_rate_DPD1":
            d1["paid_within_72h"].mean(),

        "pay_rate_DPD2":
            d2["paid_within_72h"].mean(),

        "RPM_DPD1": rpm1,
        "RPM_DPD2": rpm2,

        "diff_DPD2_minus_DPD1":
            rpm2 - rpm1,

        "uplift_DPD2_vs_DPD1":
            rpm2 / rpm1 - 1
            if rpm1 > 0 else np.nan
    })

comparison = pd.DataFrame(comparison)

print("\n" + "=" * 100)
print("DPD2 VS DPD1 — BY EXACT PAYDAY")
print("=" * 100)

display(
    comparison.style.format({
        "N_DPD1": "{:,.0f}",
        "N_DPD2": "{:,.0f}",

        "pay_rate_DPD1": "{:.1%}",
        "pay_rate_DPD2": "{:.1%}",

        "RPM_DPD1": "R$ {:,.2f}",
        "RPM_DPD2": "R$ {:,.2f}",

        "diff_DPD2_minus_DPD1": "R$ {:+,.2f}",
        "uplift_DPD2_vs_DPD1": "{:+.1%}"
    })
)

# ============================================================
# 5. CONSISTENCY
# ============================================================

comparison["DPD2_RPM_wins"] = (
    comparison["RPM_DPD2"] >
    comparison["RPM_DPD1"]
)

comparison["DPD2_payment_wins"] = (
    comparison["pay_rate_DPD2"] >
    comparison["pay_rate_DPD1"]
)

print("\n" + "=" * 100)
print("CONSISTENCY CHECK")
print("=" * 100)

print(
    "DPD2 > DPD1 in RPM:",
    f"{comparison['DPD2_RPM_wins'].sum()}/5 paydays"
)

print(
    "DPD2 > DPD1 in payment rate:",
    f"{comparison['DPD2_payment_wins'].sum()}/5 paydays"
)

N — EXACT PAYDAY × DPD FIRST MESSAGE


days_past_due,1,2,3,4,5,6,7
payday_day_of_month,,,,,,,
20,559,382,277,185,116,85,54



PAYMENT RATE 72H — EXACT PAYDAY × DPD


days_past_due,1,2,3,4,5,6,7
payday_day_of_month,,,,,,,
20,9.5%,13.1%,7.6%,8.1%,8.6%,5.9%,20.4%



RECOVERY / MESSAGE — EXACT PAYDAY × DPD


days_past_due,1,2,3,4,5,6,7
payday_day_of_month,,,,,,,
20,R$ 60.51,R$ 102.30,R$ 51.04,R$ 63.03,R$ 51.82,R$ 43.79,R$ 101.81



DPD2 VS DPD1 — BY EXACT PAYDAY


,payday,N_DPD1,N_DPD2,pay_rate_DPD1,pay_rate_DPD2,RPM_DPD1,RPM_DPD2,diff_DPD2_minus_DPD1,uplift_DPD2_vs_DPD1
0,16,0,0,nan%,nan%,R$ nan,R$ nan,R$ +nan,+nan%
1,17,0,0,nan%,nan%,R$ nan,R$ nan,R$ +nan,+nan%
2,18,0,0,nan%,nan%,R$ nan,R$ nan,R$ +nan,+nan%
3,19,0,0,nan%,nan%,R$ nan,R$ nan,R$ +nan,+nan%
4,20,559,382,9.5%,13.1%,R$ 60.51,R$ 102.30,R$ +41.79,+69.1%



CONSISTENCY CHECK
DPD2 > DPD1 in RPM: 1/5 paydays
DPD2 > DPD1 in payment rate: 1/5 paydays


In [15]:
# ============================================================
# CALENDAR TIMING ANALYSIS
# FIRST MESSAGE — DPD 1–7
#
# 1. Weekday of first message
# 2. Calendar day of first message
# 3. Weekday customer entered collections (DPD1)
# 4. Calendar day customer entered collections
# 5. DPD × weekday
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import kruskal

# ============================================================
# BASE
# ============================================================

x = first_msg.loc[
    first_msg["days_past_due"].between(1, 7)
].copy()

x["sent_at"] = pd.to_datetime(x["sent_at"])

x["days_past_due"] = pd.to_numeric(
    x["days_past_due"],
    errors="coerce"
).astype(int)

# ============================================================
# RECONSTRUCT ENTRY DATE
#
# DPD1 = first day in collections
# ============================================================

x["entry_date"] = (
    x["sent_at"].dt.normalize()
    - pd.to_timedelta(
        x["days_past_due"] - 1,
        unit="D"
    )
)

# ============================================================
# CALENDAR FEATURES
# ============================================================

weekday_map = {
    0: "Mon",
    1: "Tue",
    2: "Wed",
    3: "Thu",
    4: "Fri",
    5: "Sat",
    6: "Sun"
}

# Message date
x["message_weekday_num"] = x["sent_at"].dt.dayofweek

x["message_weekday"] = (
    x["message_weekday_num"]
    .map(weekday_map)
)

x["message_day_of_month"] = x["sent_at"].dt.day


# Entry / DPD1 date
x["entry_weekday_num"] = x["entry_date"].dt.dayofweek

x["entry_weekday"] = (
    x["entry_weekday_num"]
    .map(weekday_map)
)

x["entry_day_of_month"] = x["entry_date"].dt.day


# ============================================================
# HELPER
# ============================================================

def summarize(data, group_col):

    return (
        data.groupby(group_col)
        .agg(
            N=("customer_id", "size"),
            payers=("paid_within_72h", "sum"),
            payment_rate=("paid_within_72h", "mean"),
            recovery=("recovery_72h", "sum"),
            RPM=("recovery_72h", "mean")
        )
        .reset_index()
    )


# ============================================================
# 1. WEEKDAY OF MESSAGE
# ============================================================

weekday_order = [
    "Mon", "Tue", "Wed",
    "Thu", "Fri", "Sat", "Sun"
]

message_weekday = summarize(
    x,
    "message_weekday"
)

message_weekday["message_weekday"] = pd.Categorical(
    message_weekday["message_weekday"],
    categories=weekday_order,
    ordered=True
)

message_weekday = message_weekday.sort_values(
    "message_weekday"
)

print("=" * 100)
print("1. WEEKDAY OF FIRST MESSAGE")
print("=" * 100)

display(
    message_weekday.style.format({
        "N": "{:,.0f}",
        "payers": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery": "R$ {:,.2f}",
        "RPM": "R$ {:,.2f}"
    })
)


# ============================================================
# 2. CALENDAR DAY OF MESSAGE
# ============================================================

message_dom = summarize(
    x,
    "message_day_of_month"
)

print("\n" + "=" * 100)
print("2. CALENDAR DAY OF FIRST MESSAGE")
print("=" * 100)

display(
    message_dom.style.format({
        "N": "{:,.0f}",
        "payers": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery": "R$ {:,.2f}",
        "RPM": "R$ {:,.2f}"
    })
)


# ============================================================
# 3. WEEKDAY CUSTOMER ENTERED COLLECTIONS
# ============================================================

entry_weekday = summarize(
    x,
    "entry_weekday"
)

entry_weekday["entry_weekday"] = pd.Categorical(
    entry_weekday["entry_weekday"],
    categories=weekday_order,
    ordered=True
)

entry_weekday = entry_weekday.sort_values(
    "entry_weekday"
)

print("\n" + "=" * 100)
print("3. WEEKDAY CUSTOMER ENTERED COLLECTIONS — DPD1")
print("=" * 100)

display(
    entry_weekday.style.format({
        "N": "{:,.0f}",
        "payers": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery": "R$ {:,.2f}",
        "RPM": "R$ {:,.2f}"
    })
)


# ============================================================
# 4. CALENDAR DAY CUSTOMER ENTERED COLLECTIONS
# ============================================================

entry_dom = summarize(
    x,
    "entry_day_of_month"
)

print("\n" + "=" * 100)
print("4. CALENDAR DAY CUSTOMER ENTERED COLLECTIONS — DPD1")
print("=" * 100)

display(
    entry_dom.style.format({
        "N": "{:,.0f}",
        "payers": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery": "R$ {:,.2f}",
        "RPM": "R$ {:,.2f}"
    })
)


# ============================================================
# 5. MOST IMPORTANT:
# DPD × WEEKDAY OF MESSAGE
# ============================================================

dpd_weekday = (
    x.groupby(
        ["days_past_due", "message_weekday"]
    )
    .agg(
        N=("customer_id", "size"),
        payers=("paid_within_72h", "sum"),
        payment_rate=("paid_within_72h", "mean"),
        RPM=("recovery_72h", "mean")
    )
    .reset_index()
)

# N
N_matrix = dpd_weekday.pivot(
    index="message_weekday",
    columns="days_past_due",
    values="N"
).reindex(weekday_order)

print("\n" + "=" * 100)
print("5A. N — WEEKDAY × DPD")
print("=" * 100)

display(
    N_matrix.style.format("{:,.0f}")
)


# Payment rate
PAY_matrix = dpd_weekday.pivot(
    index="message_weekday",
    columns="days_past_due",
    values="payment_rate"
).reindex(weekday_order)

print("\n" + "=" * 100)
print("5B. PAYMENT RATE — WEEKDAY × DPD")
print("=" * 100)

display(
    PAY_matrix.style
    .format("{:.1%}")
    .background_gradient(axis=None)
)


# Recovery/message
RPM_matrix = dpd_weekday.pivot(
    index="message_weekday",
    columns="days_past_due",
    values="RPM"
).reindex(weekday_order)

print("\n" + "=" * 100)
print("5C. RECOVERY / MESSAGE — WEEKDAY × DPD")
print("=" * 100)

display(
    RPM_matrix.style
    .format("R$ {:,.2f}")
    .background_gradient(axis=None)
)


# ============================================================
# 6. BEST OBSERVED COMBINATIONS
# Only cells N >= 100
# ============================================================

best = (
    dpd_weekday.loc[
        dpd_weekday["N"] >= 100
    ]
    .sort_values(
        "RPM",
        ascending=False
    )
)

print("\n" + "=" * 100)
print("6. BEST DPD × WEEKDAY COMBINATIONS — N >= 100")
print("=" * 100)

display(
    best.head(20).style.format({
        "N": "{:,.0f}",
        "payers": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "RPM": "R$ {:,.2f}"
    })
)


1. WEEKDAY OF FIRST MESSAGE


,message_weekday,N,payers,payment_rate,recovery,RPM
1,Mon,"2,384",229,9.61%,"R$ 149,739.16",R$ 62.81
5,Tue,"1,969",215,10.92%,"R$ 138,676.19",R$ 70.43
6,Wed,"1,750",151,8.63%,"R$ 102,708.14",R$ 58.69
4,Thu,"1,794",173,9.64%,"R$ 112,844.16",R$ 62.90
0,Fri,"1,609",165,10.25%,"R$ 108,987.40",R$ 67.74
2,Sat,886,78,8.80%,"R$ 48,817.21",R$ 55.10
3,Sun,627,70,11.16%,"R$ 51,955.95",R$ 82.86



2. CALENDAR DAY OF FIRST MESSAGE


,message_day_of_month,N,payers,payment_rate,recovery,RPM
0,1,246,24,9.76%,"R$ 14,372.16",R$ 58.42
1,2,261,18,6.90%,"R$ 17,845.08",R$ 68.37
2,3,384,31,8.07%,"R$ 19,671.29",R$ 51.23
3,4,317,27,8.52%,"R$ 17,954.67",R$ 56.64
4,5,320,40,12.50%,"R$ 24,343.41",R$ 76.07
5,6,376,37,9.84%,"R$ 19,443.98",R$ 51.71
6,7,324,34,10.49%,"R$ 20,893.08",R$ 64.48
7,8,396,36,9.09%,"R$ 28,325.35",R$ 71.53
8,9,325,32,9.85%,"R$ 17,362.52",R$ 53.42
9,10,452,42,9.29%,"R$ 29,938.86",R$ 66.24



3. WEEKDAY CUSTOMER ENTERED COLLECTIONS — DPD1


...

days_past_due,1,2,3,4,5,6,7
message_weekday,,,,,,,
Mon,8.3%,10.3%,8.9%,12.4%,11.1%,9.1%,5.3%
Tue,12.5%,10.3%,11.6%,9.7%,9.4%,8.3%,8.1%
Wed,8.4%,8.1%,10.2%,9.5%,9.3%,6.7%,7.3%
Thu,9.8%,9.6%,8.1%,11.5%,8.7%,11.6%,7.9%
Fri,10.9%,11.6%,11.0%,6.3%,5.6%,6.8%,14.8%
Sat,9.2%,7.3%,7.9%,13.3%,7.0%,10.0%,8.7%
Sun,12.3%,12.0%,10.5%,7.5%,5.3%,6.7%,25.0%



5C. RECOVERY / MESSAGE — WEEKDAY × DPD


days_past_due,1,2,3,4,5,6,7
message_weekday,,,,,,,
Mon,R$ 48.19,R$ 71.34,R$ 60.44,R$ 84.37,R$ 69.37,R$ 69.18,R$ 22.58
Tue,R$ 84.13,R$ 61.43,R$ 72.10,R$ 73.75,R$ 47.22,R$ 55.03,R$ 40.29
Wed,R$ 61.57,R$ 51.31,R$ 63.83,R$ 77.95,R$ 55.64,R$ 47.20,R$ 23.35
Thu,R$ 69.47,R$ 67.20,R$ 50.83,R$ 71.93,R$ 64.59,R$ 43.25,R$ 29.22
Fri,R$ 75.59,R$ 78.05,R$ 60.60,R$ 36.45,R$ 55.84,R$ 32.51,R$ 95.82
Sat,R$ 51.97,R$ 53.44,R$ 46.34,R$ 85.66,R$ 43.54,R$ 54.67,R$ 85.04
Sun,R$ 78.88,R$ 105.74,R$ 90.77,R$ 49.51,R$ 41.91,R$ 14.95,R$ 188.94



6. BEST DPD × WEEKDAY COMBINATIONS — N >= 100


,days_past_due,message_weekday,N,payers,payment_rate,RPM
10,2,Sun,167,20,11.98%,R$ 105.74
22,4,Mon,306,38,12.42%,R$ 84.37
5,1,Tue,664,83,12.50%,R$ 84.13
3,1,Sun,228,28,12.28%,R$ 78.88
7,2,Fri,372,43,11.56%,R$ 78.05
27,4,Wed,189,18,9.52%,R$ 77.95
0,1,Fri,626,68,10.86%,R$ 75.59
26,4,Tue,278,27,9.71%,R$ 73.75
19,3,Tue,327,38,11.62%,R$ 72.10
25,4,Thu,174,20,11.49%,R$ 71.93


In [16]:
# ============================================================
# OPERATIONAL TIMING POLICY
# ENTRY WEEKDAY × DPD 1–7
# FIRST MESSAGE ONLY
# ============================================================

# ------------------------------------------------------------
# 1. Summary
# ------------------------------------------------------------

policy = (
    x.groupby(
        ["entry_weekday", "days_past_due"],
        observed=True
    )
    .agg(
        N=("customer_id", "size"),
        payers=("paid_within_72h", "sum"),
        payment_rate=("paid_within_72h", "mean"),
        recovery=("recovery_72h", "sum"),
        RPM=("recovery_72h", "mean"),
        message_weekday=("message_weekday", "first")
    )
    .reset_index()
)

# proper weekday order
policy["entry_weekday"] = pd.Categorical(
    policy["entry_weekday"],
    categories=weekday_order,
    ordered=True
)

policy = policy.sort_values(
    ["entry_weekday", "days_past_due"]
)

# ============================================================
# 2. N MATRIX
# ============================================================

N = policy.pivot(
    index="entry_weekday",
    columns="days_past_due",
    values="N"
).reindex(weekday_order)

print("=" * 110)
print("N — ENTRY WEEKDAY × DPD OF FIRST MESSAGE")
print("=" * 110)

display(
    N.style.format("{:,.0f}")
)

# ============================================================
# 3. PAYMENT RATE MATRIX
# ============================================================

PAY = policy.pivot(
    index="entry_weekday",
    columns="days_past_due",
    values="payment_rate"
).reindex(weekday_order)

print("\n" + "=" * 110)
print("PAYMENT RATE — ENTRY WEEKDAY × DPD")
print("=" * 110)

display(
    PAY.style
    .format("{:.1%}")
    .background_gradient(axis=None)
)

# ============================================================
# 4. RECOVERY / MESSAGE MATRIX
# ============================================================

RPM = policy.pivot(
    index="entry_weekday",
    columns="days_past_due",
    values="RPM"
).reindex(weekday_order)

print("\n" + "=" * 110)
print("RECOVERY / MESSAGE — ENTRY WEEKDAY × DPD")
print("=" * 110)

display(
    RPM.style
    .format("R$ {:,.2f}")
    .background_gradient(axis=None)
)

# ============================================================
# 5. WHICH WEEKDAY DOES EACH DPD LAND ON?
# ============================================================

SEND_DAY = policy.pivot(
    index="entry_weekday",
    columns="days_past_due",
    values="message_weekday"
).reindex(weekday_order)

print("\n" + "=" * 110)
print("CALENDAR MAP — ENTRY WEEKDAY → DPD → SEND WEEKDAY")
print("=" * 110)

display(SEND_DAY)

# ============================================================
# 6. BEST OBSERVED DPD BY ENTRY WEEKDAY
#
# Require N >= 100 for candidate rule
# ============================================================

candidate = policy.loc[
    policy["N"] >= 100
].copy()

best = (
    candidate
    .sort_values(
        ["entry_weekday", "RPM"],
        ascending=[True, False]
    )
    .groupby(
        "entry_weekday",
        observed=True
    )
    .first()
    .reset_index()
)

# DPD1 baseline for same entry weekday
baseline = (
    policy.loc[
        policy["days_past_due"].eq(1),
        ["entry_weekday", "RPM"]
    ]
    .rename(
        columns={"RPM": "DPD1_RPM"}
    )
)

best = best.merge(
    baseline,
    on="entry_weekday",
    how="left"
)

best["gain_vs_DPD1"] = (
    best["RPM"] -
    best["DPD1_RPM"]
)

best["uplift_vs_DPD1"] = (
    best["RPM"] /
    best["DPD1_RPM"] - 1
)

print("\n" + "=" * 110)
print("BEST OBSERVED TIMING BY ENTRY WEEKDAY — N >= 100")
print("=" * 110)

display(
    best[
        [
            "entry_weekday",
            "days_past_due",
            "message_weekday",
            "N",
            "payment_rate",
            "RPM",
            "DPD1_RPM",
            "gain_vs_DPD1",
            "uplift_vs_DPD1"
        ]
    ].style.format({
        "N": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "RPM": "R$ {:,.2f}",
        "DPD1_RPM": "R$ {:,.2f}",
        "gain_vs_DPD1": "R$ {:+,.2f}",
        "uplift_vs_DPD1": "{:+.1%}"
    })
)

N — ENTRY WEEKDAY × DPD OF FIRST MESSAGE


days_past_due,1,2,3,4,5,6,7
entry_weekday,,,,,,,
Mon,688,370,216,174,90,30,16
Tue,664,395,283,158,57,30,57
Wed,628,375,236,83,38,99,62
Thu,660,372,126,53,189,109,55
Fri,626,220,95,306,159,105,63
Sat,347,167,462,278,162,112,54
Sun,228,583,327,189,127,73,23



PAYMENT RATE — ENTRY WEEKDAY × DPD


days_past_due,1,2,3,4,5,6,7
entry_weekday,,,,,,,
Mon,8.3%,10.3%,10.2%,11.5%,5.6%,10.0%,25.0%
Tue,12.5%,8.1%,8.1%,6.3%,7.0%,6.7%,5.3%
Wed,8.4%,9.6%,11.0%,13.3%,5.3%,9.1%,8.1%
Thu,9.8%,11.6%,7.9%,7.5%,11.1%,8.3%,7.3%
Fri,10.9%,7.3%,10.5%,12.4%,9.4%,6.7%,7.9%
Sat,9.2%,12.0%,8.9%,9.7%,9.3%,11.6%,14.8%
Sun,12.3%,10.3%,11.6%,9.5%,8.7%,6.8%,8.7%



RECOVERY / MESSAGE — ENTRY WEEKDAY × DPD


days_past_due,1,2,3,4,5,6,7
entry_weekday,,,,,,,
Mon,R$ 48.19,R$ 61.43,R$ 63.83,R$ 71.93,R$ 55.84,R$ 54.67,R$ 188.94
Tue,R$ 84.13,R$ 51.31,R$ 50.83,R$ 36.45,R$ 43.54,R$ 14.95,R$ 22.58
Wed,R$ 61.57,R$ 67.20,R$ 60.60,R$ 85.66,R$ 41.91,R$ 69.18,R$ 40.29
Thu,R$ 69.47,R$ 78.05,R$ 46.34,R$ 49.51,R$ 69.37,R$ 55.03,R$ 23.35
Fri,R$ 75.59,R$ 53.44,R$ 90.77,R$ 84.37,R$ 47.22,R$ 47.20,R$ 29.22
Sat,R$ 51.97,R$ 105.74,R$ 60.44,R$ 73.75,R$ 55.64,R$ 43.25,R$ 95.82
Sun,R$ 78.88,R$ 71.34,R$ 72.10,R$ 77.95,R$ 64.59,R$ 32.51,R$ 85.04



CALENDAR MAP — ENTRY WEEKDAY → DPD → SEND WEEKDAY


days_past_due,1,2,3,4,5,6,7
entry_weekday,,,,,,,
Mon,Mon,Tue,Wed,Thu,Fri,Sat,Sun
Tue,Tue,Wed,Thu,Fri,Sat,Sun,Mon
Wed,Wed,Thu,Fri,Sat,Sun,Mon,Tue
Thu,Thu,Fri,Sat,Sun,Mon,Tue,Wed
Fri,Fri,Sat,Sun,Mon,Tue,Wed,Thu
Sat,Sat,Sun,Mon,Tue,Wed,Thu,Fri
Sun,Sun,Mon,Tue,Wed,Thu,Fri,Sat



BEST OBSERVED TIMING BY ENTRY WEEKDAY — N >= 100


,entry_weekday,days_past_due,message_weekday,N,payment_rate,RPM,DPD1_RPM,gain_vs_DPD1,uplift_vs_DPD1
0,Mon,4,Thu,174,11.49%,R$ 71.93,R$ 48.19,R$ +23.74,+49.3%
1,Tue,1,Tue,664,12.50%,R$ 84.13,R$ 84.13,R$ +0.00,+0.0%
2,Wed,2,Thu,375,9.60%,R$ 67.20,R$ 61.57,R$ +5.63,+9.2%
3,Thu,2,Fri,372,11.56%,R$ 78.05,R$ 69.47,R$ +8.58,+12.3%
4,Fri,4,Mon,306,12.42%,R$ 84.37,R$ 75.59,R$ +8.79,+11.6%
5,Sat,2,Sun,167,11.98%,R$ 105.74,R$ 51.97,R$ +53.77,+103.5%
6,Sun,1,Sun,228,12.28%,R$ 78.88,R$ 78.88,R$ +0.00,+0.0%


In [17]:
# ============================================================
# VALIDATE CANDIDATE TIMING POLICY
# Within SAME entry weekday:
# candidate DPD vs DPD1
# ============================================================

import numpy as np
import pandas as pd

from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

rng = np.random.default_rng(42)

# Candidate rules discovered above
rules = {
    "Mon": 4,
    "Wed": 2,
    "Thu": 2,
    "Fri": 4,
    "Sat": 2
}

N_PERM = 10_000
N_BOOT = 10_000

results = []

for entry_day, candidate_dpd in rules.items():

    temp = x.loc[
        x["entry_weekday"].eq(entry_day)
        & x["days_past_due"].isin([1, candidate_dpd])
    ].copy()

    control = temp.loc[
        temp["days_past_due"].eq(1),
        "recovery_72h"
    ].to_numpy()

    treatment = temp.loc[
        temp["days_past_due"].eq(candidate_dpd),
        "recovery_72h"
    ].to_numpy()

    # --------------------------------------------------------
    # Observed difference in mean recovery
    # --------------------------------------------------------

    rpm_dpd1 = control.mean()
    rpm_candidate = treatment.mean()

    diff_obs = rpm_candidate - rpm_dpd1

    uplift = (
        rpm_candidate / rpm_dpd1 - 1
        if rpm_dpd1 > 0 else np.nan
    )

    # --------------------------------------------------------
    # Permutation test
    # --------------------------------------------------------

    combined = np.concatenate([
        control,
        treatment
    ])

    n_treat = len(treatment)

    perm_diffs = np.empty(N_PERM)

    for i in range(N_PERM):

        perm = rng.permutation(combined)

        perm_diffs[i] = (
            perm[:n_treat].mean()
            -
            perm[n_treat:].mean()
        )

    p_perm = (
        np.sum(
            np.abs(perm_diffs)
            >= abs(diff_obs)
        ) + 1
    ) / (N_PERM + 1)

    # --------------------------------------------------------
    # Bootstrap CI
    # --------------------------------------------------------

    boot_diff = np.empty(N_BOOT)

    for i in range(N_BOOT):

        boot_t = rng.choice(
            treatment,
            size=len(treatment),
            replace=True
        )

        boot_c = rng.choice(
            control,
            size=len(control),
            replace=True
        )

        boot_diff[i] = (
            boot_t.mean()
            -
            boot_c.mean()
        )

    ci_low, ci_high = np.percentile(
        boot_diff,
        [2.5, 97.5]
    )

    # --------------------------------------------------------
    # Payment rate + Fisher
    # --------------------------------------------------------

    a = temp.loc[
        temp["days_past_due"].eq(candidate_dpd),
        "paid_within_72h"
    ]

    b = temp.loc[
        temp["days_past_due"].eq(1),
        "paid_within_72h"
    ]

    table = [
        [a.sum(), len(a) - a.sum()],
        [b.sum(), len(b) - b.sum()]
    ]

    _, p_fisher = fisher_exact(
        table,
        alternative="two-sided"
    )

    results.append({
        "entry_weekday": entry_day,
        "DPD1_N": len(control),
        "candidate_DPD": candidate_dpd,
        "candidate_N": len(treatment),

        "DPD1_RPM": rpm_dpd1,
        "candidate_RPM": rpm_candidate,

        "diff_RPM": diff_obs,
        "uplift": uplift,

        "CI_low": ci_low,
        "CI_high": ci_high,

        "p_perm": p_perm,

        "DPD1_payment_rate": b.mean(),
        "candidate_payment_rate": a.mean(),

        "p_fisher": p_fisher
    })


results = pd.DataFrame(results)

# ============================================================
# MULTIPLE TESTING CORRECTION
# ============================================================

results["p_perm_Holm"] = multipletests(
    results["p_perm"],
    method="holm"
)[1]

results["p_fisher_Holm"] = multipletests(
    results["p_fisher"],
    method="holm"
)[1]

results["RPM_significant"] = (
    results["p_perm_Holm"] < 0.05
)

results["payment_significant"] = (
    results["p_fisher_Holm"] < 0.05
)

# ============================================================
# OUTPUT
# ============================================================

print("=" * 130)
print("CANDIDATE TIMING VS DPD1 — SAME ENTRY WEEKDAY")
print("=" * 130)

display(
    results.style.format({
        "DPD1_N": "{:,.0f}",
        "candidate_N": "{:,.0f}",

        "DPD1_RPM": "R$ {:,.2f}",
        "candidate_RPM": "R$ {:,.2f}",
        "diff_RPM": "R$ {:+,.2f}",
        "uplift": "{:+.1%}",

        "CI_low": "R$ {:+,.2f}",
        "CI_high": "R$ {:+,.2f}",

        "p_perm": "{:.4f}",
        "p_perm_Holm": "{:.4f}",

        "DPD1_payment_rate": "{:.2%}",
        "candidate_payment_rate": "{:.2%}",

        "p_fisher": "{:.4f}",
        "p_fisher_Holm": "{:.4f}"
    })
)

CANDIDATE TIMING VS DPD1 — SAME ENTRY WEEKDAY


,entry_weekday,DPD1_N,candidate_DPD,candidate_N,DPD1_RPM,candidate_RPM,diff_RPM,uplift,CI_low,CI_high,p_perm,DPD1_payment_rate,candidate_payment_rate,p_fisher,p_perm_Holm,p_fisher_Holm,RPM_significant,payment_significant
0,Mon,688,4,174,R$ 48.19,R$ 71.93,R$ +23.74,+49.3%,R$ -14.98,R$ +67.09,0.1711,8.28%,11.49%,0.1830,0.6843,0.9151,False,False
1,Wed,628,2,375,R$ 61.57,R$ 67.20,R$ +5.63,+9.2%,R$ -25.23,R$ +37.77,0.7316,8.44%,9.60%,0.5666,1.0000,1.0000,False,False
2,Thu,660,2,372,R$ 69.47,R$ 78.05,R$ +8.58,+12.3%,R$ -25.25,R$ +43.26,0.6147,9.85%,11.56%,0.3981,1.0000,1.0000,False,False
3,Fri,626,4,306,R$ 75.59,R$ 84.37,R$ +8.79,+11.6%,R$ -27.54,R$ +46.59,0.6415,10.86%,12.42%,0.5102,1.0000,1.0000,False,False
4,Sat,347,2,167,R$ 51.97,R$ 105.74,R$ +53.77,+103.5%,R$ +1.48,R$ +112.96,0.0245,9.22%,11.98%,0.3506,0.1225,1.0000,False,False


In [19]:
# ============================================================
# WEEKDAY EFFECT CONTROLLING FOR DPD
# FIRST MESSAGE — DPD 1–7
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

z = x.copy()

# ============================================================
# 0. CLEAN TYPES
# ============================================================

# Target -> explicit 0/1
# Works for bool, 0/1, "True"/"False", etc.
if z["paid_within_72h"].dtype == bool:
    z["paid_within_72h_num"] = (
        z["paid_within_72h"].astype(int)
    )
else:
    z["paid_within_72h_num"] = (
        z["paid_within_72h"]
        .replace({
            True: 1,
            False: 0,
            "True": 1,
            "False": 0,
            "true": 1,
            "false": 0,
            "YES": 1,
            "NO": 0,
            "yes": 1,
            "no": 0
        })
    )

z["paid_within_72h_num"] = pd.to_numeric(
    z["paid_within_72h_num"],
    errors="coerce"
)

z["recovery_72h"] = pd.to_numeric(
    z["recovery_72h"],
    errors="coerce"
)

z["days_past_due"] = pd.to_numeric(
    z["days_past_due"],
    errors="coerce"
)

# Weekday categorical
weekdays = [
    "Mon", "Tue", "Wed",
    "Thu", "Fri", "Sat", "Sun"
]

z["message_weekday"] = pd.Categorical(
    z["message_weekday"],
    categories=weekdays
)

# Remove invalid rows
z = z.dropna(
    subset=[
        "paid_within_72h_num",
        "recovery_72h",
        "days_past_due",
        "message_weekday"
    ]
).copy()

z["paid_within_72h_num"] = (
    z["paid_within_72h_num"].astype(int)
)

z["days_past_due"] = (
    z["days_past_due"].astype(int)
)

# QA
print("=" * 100)
print("QA")
print("=" * 100)

print("Rows:", len(z))

print(
    "\nTarget:"
)

print(
    z["paid_within_72h_num"]
    .value_counts(dropna=False)
    .sort_index()
)

print(
    "\nWeekday:"
)

print(
    z["message_weekday"]
    .value_counts()
    .reindex(weekdays)
)

print(
    "\nDPD:"
)

print(
    z["days_past_due"]
    .value_counts()
    .sort_index()
)


# ============================================================
# 1. PAYMENT PROBABILITY
# LOGISTIC REGRESSION
# Controls for DPD
# ============================================================

m_pay = smf.logit(
    """
    paid_within_72h_num
    ~ C(message_weekday, Treatment(reference='Mon'))
    + C(days_past_due, Treatment(reference=1))
    """,
    data=z
).fit(
    cov_type="HC3",
    disp=False
)

print("\n" + "=" * 100)
print("PAYMENT MODEL")
print("WEEKDAY EFFECT CONTROLLING FOR DPD")
print("=" * 100)

pay_coef = pd.DataFrame({
    "coef_log_odds": m_pay.params,
    "odds_ratio": np.exp(m_pay.params),
    "p_value": m_pay.pvalues,
    "CI_OR_low": np.exp(m_pay.conf_int()[0]),
    "CI_OR_high": np.exp(m_pay.conf_int()[1])
})

display(
    pay_coef.style.format({
        "coef_log_odds": "{:+.3f}",
        "odds_ratio": "{:.3f}",
        "p_value": "{:.4f}",
        "CI_OR_low": "{:.3f}",
        "CI_OR_high": "{:.3f}"
    })
)


# ============================================================
# 2. RECOVERY / MESSAGE
# OLS + ROBUST SE
#
# Coefficient:
# R$ difference/message vs Monday,
# controlling for DPD
# ============================================================

m_rpm = smf.ols(
    """
    recovery_72h
    ~ C(message_weekday, Treatment(reference='Mon'))
    + C(days_past_due, Treatment(reference=1))
    """,
    data=z
).fit(
    cov_type="HC3"
)

rpm_coef = pd.DataFrame({
    "coef": m_rpm.params,
    "SE": m_rpm.bse,
    "p_value": m_rpm.pvalues,
    "CI_low": m_rpm.conf_int()[0],
    "CI_high": m_rpm.conf_int()[1]
})

print("\n" + "=" * 100)
print("RECOVERY / MESSAGE")
print("WEEKDAY EFFECT CONTROLLING FOR DPD")
print("=" * 100)

display(
    rpm_coef.style.format({
        "coef": "R$ {:+,.2f}",
        "SE": "R$ {:,.2f}",
        "p_value": "{:.4f}",
        "CI_low": "R$ {:+,.2f}",
        "CI_high": "R$ {:+,.2f}"
    })
)


# ============================================================
# 3. STANDARDIZED / ADJUSTED RPM BY WEEKDAY
#
# Every weekday evaluated using EXACTLY
# the same historical DPD distribution.
# ============================================================

adjusted = []

for wd in weekdays:

    tmp = z.copy()

    tmp["message_weekday"] = pd.Categorical(
        [wd] * len(tmp),
        categories=weekdays
    )

    pred = m_rpm.predict(tmp)

    adjusted.append({
        "weekday": wd,
        "adjusted_RPM": pred.mean()
    })

adjusted = pd.DataFrame(adjusted)

monday_rpm = adjusted.loc[
    adjusted["weekday"].eq("Mon"),
    "adjusted_RPM"
].iloc[0]

adjusted["diff_vs_Monday"] = (
    adjusted["adjusted_RPM"]
    - monday_rpm
)

adjusted["uplift_vs_Monday"] = (
    adjusted["adjusted_RPM"]
    / monday_rpm - 1
)

adjusted = adjusted.sort_values(
    "adjusted_RPM",
    ascending=False
)

print("\n" + "=" * 100)
print("ADJUSTED RECOVERY / MESSAGE BY WEEKDAY")
print("Same DPD distribution for every weekday")
print("=" * 100)

display(
    adjusted.style.format({
        "adjusted_RPM": "R$ {:,.2f}",
        "diff_vs_Monday": "R$ {:+,.2f}",
        "uplift_vs_Monday": "{:+.1%}"
    })
)

QA
Rows: 11019

Target:
paid_within_72h_num
0    9938
1    1081
Name: count, dtype: int64

Weekday:
message_weekday
Mon    2384
Tue    1969
Wed    1750
Thu    1794
Fri    1609
Sat     886
Sun     627
Name: count, dtype: int64

DPD:
days_past_due
1    3841
2    2482
3    1745
4    1241
5     822
6     558
7     330
Name: count, dtype: int64

PAYMENT MODEL
WEEKDAY EFFECT CONTROLLING FOR DPD


,coef_log_odds,odds_ratio,p_value,CI_OR_low,CI_OR_high
Intercept,-2.214,0.109,0.0000,0.093,0.128
"C(message_weekday, Treatment(reference='Mon'))[T.Tue]",+0.143,1.154,0.1529,0.948,1.405
"C(message_weekday, Treatment(reference='Mon'))[T.Wed]",-0.115,0.891,0.2953,0.718,1.106
"C(message_weekday, Treatment(reference='Mon'))[T.Thu]",+0.006,1.006,0.9532,0.818,1.239
"C(message_weekday, Treatment(reference='Mon'))[T.Fri]",+0.070,1.072,0.5174,0.868,1.324
"C(message_weekday, Treatment(reference='Mon'))[T.Sat]",-0.100,0.905,0.4663,0.691,1.185
"C(message_weekday, Treatment(reference='Mon'))[T.Sun]",+0.166,1.181,0.2496,0.890,1.567
"C(days_past_due, Treatment(reference=1))[T.2]",-0.017,0.983,0.8405,0.830,1.163
"C(days_past_due, Treatment(reference=1))[T.3]",-0.040,0.961,0.6823,0.794,1.163
"C(days_past_due, Treatment(reference=1))[T.4]",+0.024,1.024,0.8275,0.829,1.265



RECOVERY / MESSAGE
WEEKDAY EFFECT CONTROLLING FOR DPD


,coef,SE,p_value,CI_low,CI_high
Intercept,R$ +64.78,R$ 5.71,0.0000,R$ +53.60,R$ +75.97
"C(message_weekday, Treatment(reference='Mon'))[T.Tue]",R$ +7.86,R$ 7.44,0.2907,R$ -6.72,R$ +22.45
"C(message_weekday, Treatment(reference='Mon'))[T.Wed]",R$ -3.77,R$ 7.52,0.6164,R$ -18.51,R$ +10.98
"C(message_weekday, Treatment(reference='Mon'))[T.Thu]",R$ +0.57,R$ 7.48,0.9388,R$ -14.09,R$ +15.24
"C(message_weekday, Treatment(reference='Mon'))[T.Fri]",R$ +4.85,R$ 7.93,0.5406,R$ -10.68,R$ +20.38
"C(message_weekday, Treatment(reference='Mon'))[T.Sat]",R$ -8.05,R$ 8.73,0.3562,R$ -25.16,R$ +9.06
"C(message_weekday, Treatment(reference='Mon'))[T.Sun]",R$ +20.00,R$ 13.08,0.1263,R$ -5.64,R$ +45.64
"C(days_past_due, Treatment(reference=1))[T.2]",R$ +0.98,R$ 6.47,0.8791,R$ -11.69,R$ +13.66
"C(days_past_due, Treatment(reference=1))[T.3]",R$ -4.90,R$ 6.98,0.4826,R$ -18.59,R$ +8.78
"C(days_past_due, Treatment(reference=1))[T.4]",R$ +4.78,R$ 8.51,0.5743,R$ -11.90,R$ +21.47



ADJUSTED RECOVERY / MESSAGE BY WEEKDAY
Same DPD distribution for every weekday


,weekday,adjusted_RPM,diff_vs_Monday,uplift_vs_Monday
6,Sun,R$ 82.67,R$ +20.00,+31.9%
1,Tue,R$ 70.54,R$ +7.86,+12.5%
4,Fri,R$ 67.52,R$ +4.85,+7.7%
3,Thu,R$ 63.25,R$ +0.57,+0.9%
0,Mon,R$ 62.67,R$ +0.00,+0.0%
2,Wed,R$ 58.91,R$ -3.77,-6.0%
5,Sat,R$ 54.62,R$ -8.05,-12.9%


In [20]:
# ============================================================
# FINAL TEST
# DOES WEEKDAY ADD INFORMATION AFTER CONTROLLING FOR DPD?
# ============================================================

# Restricted model:
# outcome explained only by DPD
m_rpm_restricted = smf.ols(
    """
    recovery_72h
    ~ C(days_past_due, Treatment(reference=1))
    """,
    data=z
).fit()

# Full model:
# DPD + weekday
m_rpm_full = smf.ols(
    """
    recovery_72h
    ~ C(days_past_due, Treatment(reference=1))
    + C(message_weekday, Treatment(reference='Mon'))
    """,
    data=z
).fit()

# Joint robust Wald test:
# all weekday coefficients = 0
weekday_terms = [
    "C(message_weekday, Treatment(reference='Mon'))[T.Tue] = 0",
    "C(message_weekday, Treatment(reference='Mon'))[T.Wed] = 0",
    "C(message_weekday, Treatment(reference='Mon'))[T.Thu] = 0",
    "C(message_weekday, Treatment(reference='Mon'))[T.Fri] = 0",
    "C(message_weekday, Treatment(reference='Mon'))[T.Sat] = 0",
    "C(message_weekday, Treatment(reference='Mon'))[T.Sun] = 0"
]

m_rpm_robust = m_rpm_full.get_robustcov_results(
    cov_type="HC3"
)

wald = m_rpm_robust.wald_test(
    weekday_terms,
    scalar=True
)

print("=" * 80)
print("GLOBAL WEEKDAY TEST — CONTROLLING FOR DPD")
print("=" * 80)

print(f"Statistic : {float(wald.statistic):.4f}")
print(f"P-value   : {float(wald.pvalue):.6f}")

if wald.pvalue < 0.05:
    print("\n✓ Weekday contributes significantly after controlling for DPD.")
else:
    print("\n✗ No evidence that weekday contributes after controlling for DPD.")

GLOBAL WEEKDAY TEST — CONTROLLING FOR DPD
Statistic : 1.0931
P-value   : 0.363702

✗ No evidence that weekday contributes after controlling for DPD.


In [22]:
# ============================================================
# FIRST MESSAGE — DPD 1–7
# PERFORMANCE BY SEND TIME
# SELF-CONTAINED VERSION
# ============================================================

import pandas as pd
import numpy as np

h = x.copy()

# ============================================================
# 0. CLEAN / CREATE VARIABLES
# ============================================================

h["sent_at"] = pd.to_datetime(h["sent_at"])

# Target explicit 0/1
if h["paid_within_72h"].dtype == bool:
    h["paid_within_72h_num"] = (
        h["paid_within_72h"].astype(int)
    )
else:
    h["paid_within_72h_num"] = (
        h["paid_within_72h"]
        .replace({
            True: 1,
            False: 0,
            "True": 1,
            "False": 0,
            "true": 1,
            "false": 0,
            "YES": 1,
            "NO": 0,
            "yes": 1,
            "no": 0
        })
    )

h["paid_within_72h_num"] = pd.to_numeric(
    h["paid_within_72h_num"],
    errors="coerce"
)

h["recovery_72h"] = pd.to_numeric(
    h["recovery_72h"],
    errors="coerce"
)

h["days_past_due"] = pd.to_numeric(
    h["days_past_due"],
    errors="coerce"
)

# Hour
h["send_hour"] = h["sent_at"].dt.hour

# ============================================================
# 1. TIME BUCKETS
# ============================================================

bins = [-1, 7, 11, 14, 17, 20, 23]

labels = [
    "00–07",
    "08–11",
    "12–14",
    "15–17",
    "18–20",
    "21–23"
]

h["time_bucket"] = pd.cut(
    h["send_hour"],
    bins=bins,
    labels=labels
)

# ============================================================
# 2. QA
# ============================================================

print("=" * 100)
print("QA")
print("=" * 100)

print(f"Rows: {len(h):,}")

print("\nHours observed:")
print(
    h["send_hour"]
    .value_counts()
    .sort_index()
)

print("\nTime buckets:")
print(
    h["time_bucket"]
    .value_counts()
    .sort_index()
)

# ============================================================
# 3. PERFORMANCE BY TIME BUCKET
# ============================================================

time_summary = (
    h.groupby("time_bucket", observed=True)
    .agg(
        N=("customer_id", "size"),
        payers=("paid_within_72h_num", "sum"),
        payment_rate=("paid_within_72h_num", "mean"),
        recovery=("recovery_72h", "sum"),
        RPM=("recovery_72h", "mean"),
        avg_balance=("outstanding_balance_brl", "mean")
    )
    .reset_index()
)

print("\n" + "=" * 100)
print("FIRST MESSAGE — DPD1–7 — PERFORMANCE BY SEND TIME")
print("=" * 100)

display(
    time_summary.style.format({
        "N": "{:,.0f}",
        "payers": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery": "R$ {:,.2f}",
        "RPM": "R$ {:,.2f}",
        "avg_balance": "R$ {:,.2f}"
    })
)

# ============================================================
# 4. EXACT HOUR
# ============================================================

hour_summary = (
    h.groupby("send_hour")
    .agg(
        N=("customer_id", "size"),
        payers=("paid_within_72h_num", "sum"),
        payment_rate=("paid_within_72h_num", "mean"),
        recovery=("recovery_72h", "sum"),
        RPM=("recovery_72h", "mean"),
        avg_balance=("outstanding_balance_brl", "mean")
    )
    .reset_index()
)

print("\n" + "=" * 100)
print("EXACT HOUR — DIAGNOSTIC")
print("=" * 100)

display(
    hour_summary.style.format({
        "N": "{:,.0f}",
        "payers": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery": "R$ {:,.2f}",
        "RPM": "R$ {:,.2f}",
        "avg_balance": "R$ {:,.2f}"
    })
)

# ============================================================
# 5. N — TIME BUCKET × DPD
# ============================================================

N_matrix = pd.pivot_table(
    h,
    index="time_bucket",
    columns="days_past_due",
    values="customer_id",
    aggfunc="count",
    observed=True
)

print("\n" + "=" * 100)
print("N — TIME BUCKET × DPD")
print("=" * 100)

display(
    N_matrix.style.format("{:,.0f}")
)

# ============================================================
# 6. PAYMENT RATE — TIME BUCKET × DPD
# ============================================================

pay_matrix = pd.pivot_table(
    h,
    index="time_bucket",
    columns="days_past_due",
    values="paid_within_72h_num",
    aggfunc="mean",
    observed=True
)

print("\n" + "=" * 100)
print("PAYMENT RATE — TIME BUCKET × DPD")
print("=" * 100)

display(
    pay_matrix.style
    .format("{:.1%}")
    .background_gradient(axis=None)
)

# ============================================================
# 7. RECOVERY / MESSAGE — TIME BUCKET × DPD
# ============================================================

rpm_matrix = pd.pivot_table(
    h,
    index="time_bucket",
    columns="days_past_due",
    values="recovery_72h",
    aggfunc="mean",
    observed=True
)

print("\n" + "=" * 100)
print("RECOVERY / MESSAGE — TIME BUCKET × DPD")
print("=" * 100)

display(
    rpm_matrix.style
    .format("R$ {:,.2f}")
    .background_gradient(axis=None)
)

QA
Rows: 11,019

Hours observed:
send_hour
9     1483
10    1508
11    1517
12    1189
13     738
14     761
15     769
16     724
17     771
18     545
19     517
20     497
Name: count, dtype: int64

Time buckets:
time_bucket
00–07       0
08–11    4508
12–14    2688
15–17    2264
18–20    1559
21–23       0
Name: count, dtype: int64

FIRST MESSAGE — DPD1–7 — PERFORMANCE BY SEND TIME


,time_bucket,N,payers,payment_rate,recovery,RPM,avg_balance
0,08–11,"4,508",454,10.07%,"R$ 285,685.37",R$ 63.37,R$ 848.28
1,12–14,"2,688",277,10.31%,"R$ 184,556.57",R$ 68.66,R$ 844.09
2,15–17,"2,264",199,8.79%,"R$ 136,488.18",R$ 60.29,R$ 847.69
3,18–20,"1,559",151,9.69%,"R$ 106,998.09",R$ 68.63,R$ 869.08



EXACT HOUR — DIAGNOSTIC


,send_hour,N,payers,payment_rate,recovery,RPM,avg_balance
0,9,"1,483",132,8.90%,"R$ 90,329.62",R$ 60.91,R$ 850.18
1,10,"1,508",147,9.75%,"R$ 87,595.22",R$ 58.09,R$ 850.29
2,11,"1,517",175,11.54%,"R$ 107,760.53",R$ 71.04,R$ 844.43
3,12,"1,189",132,11.10%,"R$ 86,724.28",R$ 72.94,R$ 835.81
4,13,738,66,8.94%,"R$ 43,715.67",R$ 59.24,R$ 828.33
5,14,761,79,10.38%,"R$ 54,116.62",R$ 71.11,R$ 872.33
6,15,769,57,7.41%,"R$ 42,887.50",R$ 55.77,R$ 869.30
7,16,724,72,9.94%,"R$ 41,858.48",R$ 57.82,R$ 831.75
8,17,771,70,9.08%,"R$ 51,742.20",R$ 67.11,R$ 841.10
9,18,545,54,9.91%,"R$ 35,926.80",R$ 65.92,R$ 861.88



N — TIME BUCKET × DPD


days_past_due,1,2,3,4,5,6,7
time_bucket,,,,,,,
08–11,"1,562",998,732,525,332,233,126
12–14,935,594,428,313,195,131,92
15–17,803,539,331,243,170,110,68
18–20,541,351,254,160,125,84,44



PAYMENT RATE — TIME BUCKET × DPD


days_past_due,1,2,3,4,5,6,7
time_bucket,,,,,,,
08–11,10.1%,10.6%,8.9%,11.6%,10.5%,7.3%,9.5%
12–14,10.3%,9.8%,10.7%,11.5%,10.3%,8.4%,10.9%
15–17,9.5%,8.2%,9.1%,8.2%,5.9%,10.9%,10.3%
18–20,10.4%,10.5%,11.4%,6.9%,6.4%,9.5%,4.5%



RECOVERY / MESSAGE — TIME BUCKET × DPD


days_past_due,1,2,3,4,5,6,7
time_bucket,,,,,,,
08–11,R$ 66.35,R$ 68.49,R$ 49.69,R$ 79.53,R$ 60.82,R$ 37.32,R$ 52.92
12–14,R$ 68.33,R$ 60.25,R$ 77.75,R$ 79.42,R$ 74.64,R$ 48.31,R$ 63.69
15–17,R$ 64.94,R$ 61.05,R$ 59.41,R$ 55.75,R$ 36.79,R$ 81.63,R$ 43.94
18–20,R$ 68.73,R$ 88.88,R$ 75.29,R$ 55.64,R$ 47.48,R$ 36.99,R$ 35.16


In [23]:
# ============================================================
# SEND TIME — REFINED BUCKETS
# FIRST MESSAGE / DPD 1–7
# Adjusted for DPD + weekday + template
# ============================================================

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

t = h.copy()

# ============================================================
# 1. REFINED TIME BUCKET
# ============================================================

def time_bucket(hour):
    if hour <= 10:
        return "09–10"
    elif hour <= 12:
        return "11–12"
    elif hour <= 14:
        return "13–14"
    elif hour <= 17:
        return "15–17"
    else:
        return "18–20"

t["time_bucket2"] = (
    t["send_hour"]
    .apply(time_bucket)
)

bucket_order = [
    "09–10",
    "11–12",
    "13–14",
    "15–17",
    "18–20"
]

t["time_bucket2"] = pd.Categorical(
    t["time_bucket2"],
    categories=bucket_order,
    ordered=True
)

# ============================================================
# 2. RAW RESULTS
# ============================================================

raw = (
    t.groupby("time_bucket2", observed=True)
    .agg(
        N=("customer_id", "size"),
        payers=("paid_within_72h_num", "sum"),
        payment_rate=("paid_within_72h_num", "mean"),
        recovery=("recovery_72h", "sum"),
        RPM=("recovery_72h", "mean"),
        avg_balance=("outstanding_balance_brl", "mean")
    )
    .reset_index()
)

print("=" * 110)
print("RAW PERFORMANCE — REFINED TIME BUCKETS")
print("=" * 110)

display(
    raw.style.format({
        "N": "{:,.0f}",
        "payers": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery": "R$ {:,.2f}",
        "RPM": "R$ {:,.2f}",
        "avg_balance": "R$ {:,.2f}"
    })
)

# ============================================================
# 3. PAYMENT MODEL
# time + DPD + weekday + template
# ============================================================

m_pay_time = smf.logit(
    """
    paid_within_72h_num
    ~ C(time_bucket2, Treatment(reference='09–10'))
    + C(days_past_due)
    + C(message_weekday)
    + C(template)
    """,
    data=t
).fit(
    cov_type="HC3",
    disp=False
)

pay_table = pd.DataFrame({
    "OR": np.exp(m_pay_time.params),
    "p_value": m_pay_time.pvalues,
    "CI_low": np.exp(m_pay_time.conf_int()[0]),
    "CI_high": np.exp(m_pay_time.conf_int()[1])
})

print("\n" + "=" * 110)
print("PAYMENT PROBABILITY — ADJUSTED")
print("=" * 110)

display(
    pay_table.loc[
        pay_table.index.str.contains("time_bucket2")
    ].style.format({
        "OR": "{:.3f}",
        "p_value": "{:.4f}",
        "CI_low": "{:.3f}",
        "CI_high": "{:.3f}"
    })
)

# ============================================================
# 4. RECOVERY / MESSAGE MODEL
# ============================================================

m_rpm_time = smf.ols(
    """
    recovery_72h
    ~ C(time_bucket2, Treatment(reference='09–10'))
    + C(days_past_due)
    + C(message_weekday)
    + C(template)
    """,
    data=t
).fit(
    cov_type="HC3"
)

rpm_table = pd.DataFrame({
    "coef": m_rpm_time.params,
    "SE": m_rpm_time.bse,
    "p_value": m_rpm_time.pvalues,
    "CI_low": m_rpm_time.conf_int()[0],
    "CI_high": m_rpm_time.conf_int()[1]
})

print("\n" + "=" * 110)
print("RECOVERY / MESSAGE — ADJUSTED")
print("=" * 110)

display(
    rpm_table.loc[
        rpm_table.index.str.contains("time_bucket2")
    ].style.format({
        "coef": "R$ {:+,.2f}",
        "SE": "R$ {:,.2f}",
        "p_value": "{:.4f}",
        "CI_low": "R$ {:+,.2f}",
        "CI_high": "R$ {:+,.2f}"
    })
)

# ============================================================
# 5. GLOBAL TEST
#
# H0: time bucket adds no information
# ============================================================

terms = [
    "C(time_bucket2, Treatment(reference='09–10'))[T.11–12] = 0",
    "C(time_bucket2, Treatment(reference='09–10'))[T.13–14] = 0",
    "C(time_bucket2, Treatment(reference='09–10'))[T.15–17] = 0",
    "C(time_bucket2, Treatment(reference='09–10'))[T.18–20] = 0"
]

wald = m_rpm_time.wald_test(
    terms,
    scalar=True
)

print("\n" + "=" * 110)
print("GLOBAL TIME-BUCKET TEST")
print("=" * 110)

print(
    f"Statistic : {float(wald.statistic):.4f}"
)

print(
    f"P-value   : {float(wald.pvalue):.6f}"
)

# ============================================================
# 6. ADJUSTED RPM
#
# Standardize every customer into each time bucket
# ============================================================

adjusted = []

for bucket in bucket_order:

    tmp = t.copy()

    tmp["time_bucket2"] = pd.Categorical(
        [bucket] * len(tmp),
        categories=bucket_order,
        ordered=True
    )

    pred = m_rpm_time.predict(tmp)

    adjusted.append({
        "time_bucket": bucket,
        "adjusted_RPM": pred.mean()
    })

adjusted = pd.DataFrame(adjusted)

baseline = adjusted.loc[
    adjusted["time_bucket"].eq("09–10"),
    "adjusted_RPM"
].iloc[0]

adjusted["diff_vs_09_10"] = (
    adjusted["adjusted_RPM"]
    - baseline
)

adjusted["uplift_vs_09_10"] = (
    adjusted["adjusted_RPM"]
    / baseline - 1
)

adjusted = adjusted.sort_values(
    "adjusted_RPM",
    ascending=False
)

print("\n" + "=" * 110)
print("ADJUSTED RECOVERY / MESSAGE BY SEND TIME")
print("=" * 110)

display(
    adjusted.style.format({
        "adjusted_RPM": "R$ {:,.2f}",
        "diff_vs_09_10": "R$ {:+,.2f}",
        "uplift_vs_09_10": "{:+.1%}"
    })
)

RAW PERFORMANCE — REFINED TIME BUCKETS


,time_bucket2,N,payers,payment_rate,recovery,RPM,avg_balance
0,09–10,"2,991",279,9.33%,"R$ 177,924.84",R$ 59.49,R$ 850.24
1,11–12,"2,706",307,11.35%,"R$ 194,484.81",R$ 71.87,R$ 840.64
2,13–14,"1,499",145,9.67%,"R$ 97,832.29",R$ 65.27,R$ 850.67
3,15–17,"2,264",199,8.79%,"R$ 136,488.18",R$ 60.29,R$ 847.69
4,18–20,"1,559",151,9.69%,"R$ 106,998.09",R$ 68.63,R$ 869.08



PAYMENT PROBABILITY — ADJUSTED


,OR,p_value,CI_low,CI_high
"C(time_bucket2, Treatment(reference='09–10'))[T.11–12]",1.244,0.0126,1.048,1.476
"C(time_bucket2, Treatment(reference='09–10'))[T.13–14]",1.040,0.7166,0.842,1.285
"C(time_bucket2, Treatment(reference='09–10'))[T.15–17]",0.937,0.5032,0.774,1.134
"C(time_bucket2, Treatment(reference='09–10'))[T.18–20]",1.046,0.6694,0.850,1.289



RECOVERY / MESSAGE — ADJUSTED


,coef,SE,p_value,CI_low,CI_high
"C(time_bucket2, Treatment(reference='09–10'))[T.11–12]",R$ +12.41,R$ 6.47,0.0550,R$ -0.27,R$ +25.08
"C(time_bucket2, Treatment(reference='09–10'))[T.13–14]",R$ +5.85,R$ 7.71,0.4476,R$ -9.26,R$ +20.96
"C(time_bucket2, Treatment(reference='09–10'))[T.15–17]",R$ +0.76,R$ 6.64,0.9087,R$ -12.25,R$ +13.77
"C(time_bucket2, Treatment(reference='09–10'))[T.18–20]",R$ +9.52,R$ 7.92,0.2291,R$ -5.99,R$ +25.04



GLOBAL TIME-BUCKET TEST
Statistic : 4.8937
P-value   : 0.298378

ADJUSTED RECOVERY / MESSAGE BY SEND TIME


,time_bucket,adjusted_RPM,diff_vs_09_10,uplift_vs_09_10
1,11–12,R$ 71.83,R$ +12.41,+20.9%
4,18–20,R$ 68.95,R$ +9.52,+16.0%
2,13–14,R$ 65.28,R$ +5.85,+9.9%
3,15–17,R$ 60.19,R$ +0.76,+1.3%
0,09–10,R$ 59.43,R$ +0.00,+0.0%


In [24]:
# ============================================================
# HYPOTHESIS TEST
# DPD2 + 18–20h vs ALL OTHER DPD × TIME COMBINATIONS
# FIRST MESSAGE ONLY
# ============================================================

import pandas as pd
import numpy as np
from scipy.stats import fisher_exact

d = h.copy()

# ------------------------------------------------------------
# 1. TARGET CELL
# ------------------------------------------------------------

d["target_dpd2_18_20"] = (
    (d["days_past_due"] == 2)
    & (d["send_hour"].between(18, 20))
).astype(int)

# ------------------------------------------------------------
# 2. RAW COMPARISON
# ------------------------------------------------------------

raw = (
    d.groupby("target_dpd2_18_20")
    .agg(
        N=("customer_id", "size"),
        payers=("paid_within_72h_num", "sum"),
        payment_rate=("paid_within_72h_num", "mean"),
        recovery=("recovery_72h", "sum"),
        RPM=("recovery_72h", "mean"),
        avg_balance=("outstanding_balance_brl", "mean")
    )
)

raw.index = ["All other combinations", "DPD2 + 18–20h"]

print("=" * 100)
print("DPD2 + 18–20h vs ALL OTHER COMBINATIONS")
print("=" * 100)

display(
    raw.style.format({
        "N": "{:,.0f}",
        "payers": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "recovery": "R$ {:,.2f}",
        "RPM": "R$ {:,.2f}",
        "avg_balance": "R$ {:,.2f}"
    })
)

# ============================================================
# 3. PERMUTATION TEST — RECOVERY / MESSAGE
# ============================================================

rng = np.random.default_rng(42)

target = d.loc[
    d["target_dpd2_18_20"].eq(1),
    "recovery_72h"
].dropna().to_numpy()

other = d.loc[
    d["target_dpd2_18_20"].eq(0),
    "recovery_72h"
].dropna().to_numpy()

observed_diff = target.mean() - other.mean()

combined = np.concatenate([target, other])
n_target = len(target)

B = 10_000
perm_diffs = np.empty(B)

for b in range(B):
    perm = rng.permutation(combined)

    perm_diffs[b] = (
        perm[:n_target].mean()
        - perm[n_target:].mean()
    )

p_perm = (
    np.sum(
        np.abs(perm_diffs)
        >= abs(observed_diff)
    ) + 1
) / (B + 1)

# ============================================================
# 4. BOOTSTRAP CI
# ============================================================

B_BOOT = 10_000
boot_diff = np.empty(B_BOOT)

for b in range(B_BOOT):

    a = rng.choice(
        target,
        size=len(target),
        replace=True
    )

    c = rng.choice(
        other,
        size=len(other),
        replace=True
    )

    boot_diff[b] = a.mean() - c.mean()

ci_low, ci_high = np.percentile(
    boot_diff,
    [2.5, 97.5]
)

# ============================================================
# 5. PAYMENT — FISHER EXACT
# ============================================================

target_pay = int(
    d.loc[
        d["target_dpd2_18_20"].eq(1),
        "paid_within_72h_num"
    ].sum()
)

target_nonpay = (
    d["target_dpd2_18_20"].sum()
    - target_pay
)

other_pay = int(
    d.loc[
        d["target_dpd2_18_20"].eq(0),
        "paid_within_72h_num"
    ].sum()
)

other_nonpay = (
    (d["target_dpd2_18_20"] == 0).sum()
    - other_pay
)

table = [
    [target_pay, target_nonpay],
    [other_pay, other_nonpay]
]

odds_ratio, p_fisher = fisher_exact(
    table,
    alternative="two-sided"
)

# ============================================================
# 6. RESULTS
# ============================================================

target_rpm = target.mean()
other_rpm = other.mean()

uplift = (
    target_rpm / other_rpm - 1
)

print("\n" + "=" * 100)
print("RECOVERY / MESSAGE")
print("=" * 100)

print(f"N target       : {len(target):,}")
print(f"N others       : {len(other):,}")

print(f"\nDPD2 18–20 RPM : R$ {target_rpm:,.2f}")
print(f"Others RPM     : R$ {other_rpm:,.2f}")

print(f"\nDifference     : R$ {observed_diff:+,.2f}")
print(f"Uplift         : {uplift:+.1%}")

print(
    f"Bootstrap 95% CI: "
    f"[R$ {ci_low:+,.2f}, R$ {ci_high:+,.2f}]"
)

print(f"Permutation p  : {p_perm:.6f}")

print("\n" + "=" * 100)
print("PAYMENT RATE")
print("=" * 100)

target_rate = target_pay / (
    target_pay + target_nonpay
)

other_rate = other_pay / (
    other_pay + other_nonpay
)

print(f"DPD2 18–20     : {target_rate:.2%}")
print(f"Others         : {other_rate:.2%}")

print(f"Odds ratio      : {odds_ratio:.3f}")
print(f"Fisher p-value  : {p_fisher:.6f}")

DPD2 + 18–20h vs ALL OTHER COMBINATIONS


,N,payers,payment_rate,recovery,RPM,avg_balance
All other combinations,"10,668","1,044",9.79%,"R$ 682,529.71",R$ 63.98,R$ 848.91
DPD2 + 18–20h,351,37,10.54%,"R$ 31,198.50",R$ 88.88,R$ 885.59



RECOVERY / MESSAGE
N target       : 351
N others       : 10,668

DPD2 18–20 RPM : R$ 88.88
Others RPM     : R$ 63.98

Difference     : R$ +24.91
Uplift         : +38.9%
Bootstrap 95% CI: [R$ -6.09, R$ +59.06]
Permutation p  : 0.059294

PAYMENT RATE
DPD2 18–20     : 10.54%
Others         : 9.79%
Odds ratio      : 1.086
Fisher p-value  : 0.648019


In [25]:
# ============================================================
# DPD2 ONLY
# IS 18–20h BETTER THAN OTHER HOURS?
# ============================================================

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from scipy.stats import fisher_exact

d2 = h.loc[
    h["days_past_due"].eq(2)
].copy()

d2["send_18_20"] = (
    d2["send_hour"].between(18, 20)
).astype(int)

# log balance — controls for economic exposure
d2["log_balance"] = np.log1p(
    pd.to_numeric(
        d2["outstanding_balance_brl"],
        errors="coerce"
    )
)

# ============================================================
# 1. RAW
# ============================================================

raw = (
    d2.groupby("send_18_20")
    .agg(
        N=("customer_id", "size"),
        payers=("paid_within_72h_num", "sum"),
        payment_rate=("paid_within_72h_num", "mean"),
        avg_balance=("outstanding_balance_brl", "mean"),
        recovery=("recovery_72h", "sum"),
        RPM=("recovery_72h", "mean")
    )
)

raw.index = ["Other hours", "18–20h"]

print("=" * 100)
print("DPD2 ONLY — 18–20h vs OTHER HOURS")
print("=" * 100)

display(
    raw.style.format({
        "N": "{:,.0f}",
        "payers": "{:,.0f}",
        "payment_rate": "{:.2%}",
        "avg_balance": "R$ {:,.2f}",
        "recovery": "R$ {:,.2f}",
        "RPM": "R$ {:,.2f}"
    })
)

# ============================================================
# 2. PAYMENT — ADJUSTED
# ============================================================

m_pay = smf.logit(
    """
    paid_within_72h_num
    ~ send_18_20
    + log_balance
    + C(message_weekday)
    + C(template)
    """,
    data=d2
).fit(
    cov_type="HC3",
    disp=False
)

beta = m_pay.params["send_18_20"]
ci = m_pay.conf_int().loc["send_18_20"]

print("\n" + "=" * 100)
print("PAYMENT — ADJUSTED")
print("=" * 100)

print(f"Odds ratio : {np.exp(beta):.3f}")
print(
    f"95% CI     : "
    f"[{np.exp(ci.iloc[0]):.3f}, "
    f"{np.exp(ci.iloc[1]):.3f}]"
)
print(
    f"P-value    : "
    f"{m_pay.pvalues['send_18_20']:.6f}"
)

# ============================================================
# 3. RECOVERY / MESSAGE — ADJUSTED
# ============================================================

m_rec = smf.ols(
    """
    recovery_72h
    ~ send_18_20
    + log_balance
    + C(message_weekday)
    + C(template)
    """,
    data=d2
).fit(
    cov_type="HC3"
)

coef = m_rec.params["send_18_20"]
ci = m_rec.conf_int().loc["send_18_20"]

print("\n" + "=" * 100)
print("RECOVERY / MESSAGE — ADJUSTED")
print("=" * 100)

print(f"Difference : R$ {coef:+,.2f}")
print(
    f"95% CI     : "
    f"[R$ {ci.iloc[0]:+,.2f}, "
    f"R$ {ci.iloc[1]:+,.2f}]"
)
print(
    f"P-value    : "
    f"{m_rec.pvalues['send_18_20']:.6f}"
)

# ============================================================
# 4. STANDARDIZED RPM
# ============================================================

pred_other = d2.copy()
pred_other["send_18_20"] = 0

pred_target = d2.copy()
pred_target["send_18_20"] = 1

rpm_other = m_rec.predict(pred_other).mean()
rpm_target = m_rec.predict(pred_target).mean()

print("\n" + "=" * 100)
print("STANDARDIZED RECOVERY / MESSAGE")
print("=" * 100)

print(f"Other hours : R$ {rpm_other:,.2f}")
print(f"18–20h      : R$ {rpm_target:,.2f}")
print(f"Difference  : R$ {rpm_target-rpm_other:+,.2f}")
print(
    f"Uplift      : "
    f"{rpm_target/rpm_other-1:+.1%}"
)

DPD2 ONLY — 18–20h vs OTHER HOURS


,N,payers,payment_rate,avg_balance,recovery,RPM
Other hours,"2,131",208,9.76%,R$ 845.55,"R$ 137,045.76",R$ 64.31
18–20h,351,37,10.54%,R$ 885.59,"R$ 31,198.50",R$ 88.88



PAYMENT — ADJUSTED
Odds ratio : 1.111
95% CI     : [0.766, 1.611]
P-value    : 0.578624

RECOVERY / MESSAGE — ADJUSTED
Difference : R$ +22.65
95% CI     : [R$ -10.53, R$ +55.83]
P-value    : 0.180995

STANDARDIZED RECOVERY / MESSAGE
Other hours : R$ 64.58
18–20h      : R$ 87.23
Difference  : R$ +22.65
Uplift      : +35.1%


In [27]:
# ============================================================
# UNIVERSAL SEND-TIME RULE
# Test every time bucket vs all other hours
#
# Population: first message / DPD 1–7
# Controls: DPD + weekday + template + balance
# Multiple testing: Holm
# ============================================================

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

u = h.copy()

# ------------------------------------------------------------
# 1. TIME BUCKETS
# ------------------------------------------------------------

def make_bucket(hour):
    if hour <= 10:
        return "09–10"
    elif hour <= 12:
        return "11–12"
    elif hour <= 14:
        return "13–14"
    elif hour <= 17:
        return "15–17"
    else:
        return "18–20"

u["time_bucket"] = u["send_hour"].apply(make_bucket)

buckets = [
    "09–10",
    "11–12",
    "13–14",
    "15–17",
    "18–20"
]

u["log_balance"] = np.log1p(
    pd.to_numeric(
        u["outstanding_balance_brl"],
        errors="coerce"
    )
)

# ============================================================
# 2. EACH BUCKET vs ALL OTHER HOURS
# ============================================================

results = []

for bucket in buckets:

    tmp = u.copy()

    tmp["target_time"] = (
        tmp["time_bucket"] == bucket
    ).astype(int)

    # ------------------------------------------
    # PAYMENT MODEL
    # ------------------------------------------

    m_pay = smf.logit(
        """
        paid_within_72h_num
        ~ target_time
        + C(days_past_due)
        + C(message_weekday)
        + C(template)
        + log_balance
        """,
        data=tmp
    ).fit(
        cov_type="HC3",
        disp=False
    )

    # ------------------------------------------
    # RECOVERY MODEL
    # ------------------------------------------

    m_rec = smf.ols(
        """
        recovery_72h
        ~ target_time
        + C(days_past_due)
        + C(message_weekday)
        + C(template)
        + log_balance
        """,
        data=tmp
    ).fit(
        cov_type="HC3"
    )

    # ------------------------------------------
    # STANDARDIZED RECOVERY
    # ------------------------------------------

    target_df = tmp.copy()
    other_df = tmp.copy()

    target_df["target_time"] = 1
    other_df["target_time"] = 0

    rpm_target = m_rec.predict(target_df).mean()
    rpm_other = m_rec.predict(other_df).mean()

    # ------------------------------------------
    # RAW
    # ------------------------------------------

    mask = tmp["target_time"].eq(1)

    n = mask.sum()

    raw_pay = tmp.loc[
        mask,
        "paid_within_72h_num"
    ].mean()

    raw_rpm = tmp.loc[
        mask,
        "recovery_72h"
    ].mean()

    # ------------------------------------------
    # CI
    # ------------------------------------------

    ci_rec = m_rec.conf_int().loc["target_time"]

    beta_pay = m_pay.params["target_time"]
    ci_pay = m_pay.conf_int().loc["target_time"]

    results.append({
        "time_bucket": bucket,
        "N": n,

        "raw_payment_rate": raw_pay,
        "raw_RPM": raw_rpm,

        "OR_payment": np.exp(beta_pay),
        "payment_CI_low": np.exp(ci_pay.iloc[0]),
        "payment_CI_high": np.exp(ci_pay.iloc[1]),
        "payment_p_raw": m_pay.pvalues["target_time"],

        "adjusted_RPM": rpm_target,
        "other_adjusted_RPM": rpm_other,
        "RPM_diff": rpm_target - rpm_other,
        "RPM_uplift": rpm_target / rpm_other - 1,

        "recovery_CI_low": ci_rec.iloc[0],
        "recovery_CI_high": ci_rec.iloc[1],
        "recovery_p_raw": m_rec.pvalues["target_time"]
    })

res = pd.DataFrame(results)

# ============================================================
# 3. MULTIPLE-TEST CORRECTION — HOLM
# ============================================================

res["payment_p_holm"] = multipletests(
    res["payment_p_raw"],
    method="holm"
)[1]

res["recovery_p_holm"] = multipletests(
    res["recovery_p_raw"],
    method="holm"
)[1]

# ============================================================
# 4. OUTPUT
# ============================================================

res = res.sort_values(
    "adjusted_RPM",
    ascending=False
).reset_index(drop=True)

print("=" * 130)
print("UNIVERSAL SEND-TIME TEST")
print("Each window vs ALL OTHER HOURS")
print("Adjusted for DPD + weekday + template + balance")
print("=" * 130)

display(
    res.style.format({
        "N": "{:,.0f}",

        "raw_payment_rate": "{:.2%}",
        "raw_RPM": "R$ {:,.2f}",

        "OR_payment": "{:.3f}",
        "payment_CI_low": "{:.3f}",
        "payment_CI_high": "{:.3f}",
        "payment_p_raw": "{:.4f}",
        "payment_p_holm": "{:.4f}",

        "adjusted_RPM": "R$ {:,.2f}",
        "other_adjusted_RPM": "R$ {:,.2f}",
        "RPM_diff": "R$ {:+,.2f}",
        "RPM_uplift": "{:+.1%}",

        "recovery_CI_low": "R$ {:+,.2f}",
        "recovery_CI_high": "R$ {:+,.2f}",
        "recovery_p_raw": "{:.4f}",
        "recovery_p_holm": "{:.4f}"
    })
)

# ============================================================
# 5. CONSISTENCY ACROSS DPD
# ============================================================

consistency_dpd = (
    u.groupby(
        ["days_past_due", "time_bucket"]
    )
    .agg(
        N=("customer_id", "size"),
        payment_rate=("paid_within_72h_num", "mean"),
        RPM=("recovery_72h", "mean")
    )
    .reset_index()
)

print("\n" + "=" * 130)
print("CONSISTENCY CHECK — RPM BY DPD")
print("=" * 130)

rpm_dpd = consistency_dpd.pivot(
    index="time_bucket",
    columns="days_past_due",
    values="RPM"
).reindex(buckets)

display(
    rpm_dpd.style
    .format("R$ {:,.2f}")
    .background_gradient(axis=0)
)

# ============================================================
# 6. CONSISTENCY ACROSS WEEKDAY
# ============================================================

consistency_weekday = (
    u.groupby(
        ["message_weekday", "time_bucket"]
    )
    .agg(
        N=("customer_id", "size"),
        payment_rate=("paid_within_72h_num", "mean"),
        RPM=("recovery_72h", "mean")
    )
    .reset_index()
)

weekday_order = [
    "Mon", "Tue", "Wed",
    "Thu", "Fri", "Sat", "Sun"
]

rpm_weekday = consistency_weekday.pivot(
    index="time_bucket",
    columns="message_weekday",
    values="RPM"
).reindex(
    index=buckets,
    columns=weekday_order
)

print("\n" + "=" * 130)
print("CONSISTENCY CHECK — RPM BY WEEKDAY")
print("=" * 130)

display(
    rpm_weekday.style
    .format("R$ {:,.2f}")
    .background_gradient(axis=0)
)

UNIVERSAL SEND-TIME TEST
Each window vs ALL OTHER HOURS
Adjusted for DPD + weekday + template + balance


,time_bucket,N,raw_payment_rate,raw_RPM,OR_payment,payment_CI_low,payment_CI_high,payment_p_raw,adjusted_RPM,other_adjusted_RPM,RPM_diff,RPM_uplift,recovery_CI_low,recovery_CI_high,recovery_p_raw,payment_p_holm,recovery_p_holm
0,11–12,"2,706",11.35%,R$ 71.87,1.239,1.077,1.425,0.0027,R$ 72.57,R$ 62.24,R$ +10.33,+16.6%,R$ -0.36,R$ +21.02,0.0583,0.0137,0.2915
1,18–20,"1,559",9.69%,R$ 68.63,1.000,0.834,1.199,0.9998,R$ 67.25,R$ 64.36,R$ +2.89,+4.5%,R$ -10.88,R$ +16.65,0.6812,1.0000,1.0000
2,13–14,"1,499",9.67%,R$ 65.27,0.982,0.816,1.181,0.8449,R$ 65.08,R$ 64.72,R$ +0.35,+0.5%,R$ -13.01,R$ +13.71,0.9590,1.0000,1.0000
3,15–17,"2,264",8.79%,R$ 60.29,0.857,0.729,1.007,0.0610,R$ 60.53,R$ 65.87,R$ -5.34,-8.1%,R$ -16.43,R$ +5.74,0.3448,0.2442,1.0000
4,09–10,"2,991",9.33%,R$ 59.49,0.926,0.802,1.069,0.2959,R$ 59.49,R$ 66.74,R$ -7.25,-10.9%,R$ -17.24,R$ +2.75,0.1553,0.8877,0.6211



CONSISTENCY CHECK — RPM BY DPD


days_past_due,1,2,3,4,5,6,7
time_bucket,,,,,,,
09–10,R$ 64.95,R$ 66.72,R$ 46.10,R$ 69.25,R$ 46.44,R$ 44.47,R$ 27.29
11–12,R$ 71.14,R$ 55.98,R$ 69.91,R$ 105.98,R$ 87.08,R$ 45.99,R$ 83.72
13–14,R$ 64.29,R$ 79.46,R$ 70.57,R$ 51.81,R$ 65.93,R$ 25.44,R$ 57.81
15–17,R$ 64.94,R$ 61.05,R$ 59.41,R$ 55.75,R$ 36.79,R$ 81.63,R$ 43.94
18–20,R$ 68.73,R$ 88.88,R$ 75.29,R$ 55.64,R$ 47.48,R$ 36.99,R$ 35.16



CONSISTENCY CHECK — RPM BY WEEKDAY


message_weekday,Mon,Tue,Wed,Thu,Fri,Sat,Sun
time_bucket,,,,,,,
09–10,R$ 60.24,R$ 68.40,R$ 45.15,R$ 54.98,R$ 60.92,R$ 50.01,R$ 93.14
11–12,R$ 61.80,R$ 80.51,R$ 92.16,R$ 76.74,R$ 59.06,R$ 52.95,R$ 69.01
13–14,R$ 55.15,R$ 64.23,R$ 51.04,R$ 94.37,R$ 72.04,R$ 61.03,R$ 48.00
15–17,R$ 72.23,R$ 63.76,R$ 50.46,R$ 42.92,R$ 61.00,R$ 53.57,R$ 85.11
18–20,R$ 62.53,R$ 71.54,R$ 41.67,R$ 55.84,R$ 101.47,R$ 64.04,R$ 115.01


In [28]:
# ============================================================
# SEND HOUR × INTERACTION
# Read / Click / Reply / Any engagement
# ============================================================

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

hist = wa.copy()

hist["sent_at"] = pd.to_datetime(hist["sent_at"])

# Histórico antes de setembro
hist = hist.loc[
    hist["sent_at"] < pd.Timestamp("2026-09-01")
].copy()

hist["send_hour"] = hist["sent_at"].dt.hour

# Mesma janela usada na plan
hist = hist.loc[
    hist["send_hour"].between(9, 20)
].copy()

# ------------------------------------------------------------
# Interaction flags
# ------------------------------------------------------------

hist["is_read"] = hist["interaction"].eq("read").astype(int)
hist["is_clicked"] = hist["interaction"].eq("clicked").astype(int)
hist["is_replied"] = hist["interaction"].eq("replied").astype(int)

hist["any_engagement"] = (
    hist["interaction"]
    .isin(["read", "clicked", "replied"])
    .astype(int)
)

# ============================================================
# 1. SAMPLE SIZE + RAW INTERACTION BY HOUR
# ============================================================

hour_interaction = (
    hist
    .groupby("send_hour")
    .agg(
        messages=("message_id", "count"),
        customers=("customer_id", "nunique"),
        reads=("is_read", "sum"),
        clicks=("is_clicked", "sum"),
        replies=("is_replied", "sum"),
        any_engagement=("any_engagement", "sum")
    )
)

for col in ["reads", "clicks", "replies", "any_engagement"]:
    hour_interaction[f"{col}_rate_pct"] = (
        hour_interaction[col]
        / hour_interaction["messages"]
        * 100
    )

print("=" * 115)
print("SEND HOUR × INTERACTION — ALL MESSAGES")
print("=" * 115)

print(
    hour_interaction[
        [
            "messages",
            "reads_rate_pct",
            "clicks_rate_pct",
            "replies_rate_pct",
            "any_engagement_rate_pct"
        ]
    ]
    .round(2)
    .to_string()
)


# ============================================================
# 2. INTERACTION DISTRIBUTION
# ============================================================

interaction_dist = pd.crosstab(
    hist["send_hour"],
    hist["interaction"]
)

interaction_pct = pd.crosstab(
    hist["send_hour"],
    hist["interaction"],
    normalize="index"
) * 100

print("\n" + "=" * 115)
print("INTERACTION COUNTS")
print("=" * 115)

print(interaction_dist.to_string())

print("\n" + "=" * 115)
print("INTERACTION DISTRIBUTION (%)")
print("=" * 115)

print(interaction_pct.round(2).to_string())


# ============================================================
# 3. GLOBAL CHI-SQUARE
# H0: interaction distribution does not depend on send hour
# ============================================================

chi2, p, dof, expected = chi2_contingency(interaction_dist)

print("\n" + "=" * 115)
print("GLOBAL TEST — HOUR × INTERACTION")
print("=" * 115)

print(f"Chi-square : {chi2:,.2f}")
print(f"DoF        : {dof}")
print(f"P-value    : {p:.8f}")

if p < 0.05:
    print("→ Significant association between send hour and interaction.")
else:
    print("→ No statistically significant association.")


# ============================================================
# 4. INDIVIDUAL TESTS
# hour × read / click / reply / engagement
# ============================================================

print("\n" + "=" * 115)
print("INDIVIDUAL GLOBAL TESTS")
print("=" * 115)

outcomes = [
    ("READ", "is_read"),
    ("CLICK", "is_clicked"),
    ("REPLY", "is_replied"),
    ("ANY ENGAGEMENT", "any_engagement"),
]

test_results = []

for label, col in outcomes:

    tab = pd.crosstab(
        hist["send_hour"],
        hist[col]
    )

    chi2_i, p_i, dof_i, _ = chi2_contingency(tab)

    test_results.append({
        "outcome": label,
        "chi2": chi2_i,
        "dof": dof_i,
        "p_value": p_i
    })

test_results = pd.DataFrame(test_results)

print(
    test_results
    .round({
        "chi2": 2,
        "p_value": 8
    })
    .to_string(index=False)
)


# ============================================================
# 5. DELIVERED ONLY
#
# Separates:
#   "Was the message delivered?"
# from
#   "Given delivery, did the customer engage?"
# ============================================================

deliv = hist.loc[
    hist["delivery_status"].eq("delivered")
].copy()

deliv_hour = (
    deliv
    .groupby("send_hour")
    .agg(
        delivered=("message_id", "count"),
        reads=("is_read", "sum"),
        clicks=("is_clicked", "sum"),
        replies=("is_replied", "sum"),
        any_engagement=("any_engagement", "sum")
    )
)

for col in ["reads", "clicks", "replies", "any_engagement"]:
    deliv_hour[f"{col}_rate_pct"] = (
        deliv_hour[col]
        / deliv_hour["delivered"]
        * 100
    )

print("\n" + "=" * 115)
print("SEND HOUR × INTERACTION — DELIVERED ONLY")
print("=" * 115)

print(
    deliv_hour[
        [
            "delivered",
            "reads_rate_pct",
            "clicks_rate_pct",
            "replies_rate_pct",
            "any_engagement_rate_pct"
        ]
    ]
    .round(2)
    .to_string()
)


# ============================================================
# 6. DELIVERED-ONLY GLOBAL TESTS
# ============================================================

print("\n" + "=" * 115)
print("DELIVERED ONLY — GLOBAL TESTS")
print("=" * 115)

deliv_tests = []

for label, col in outcomes:

    tab = pd.crosstab(
        deliv["send_hour"],
        deliv[col]
    )

    chi2_i, p_i, dof_i, _ = chi2_contingency(tab)

    deliv_tests.append({
        "outcome": label,
        "chi2": chi2_i,
        "dof": dof_i,
        "p_value": p_i
    })

deliv_tests = pd.DataFrame(deliv_tests)

print(
    deliv_tests
    .round({
        "chi2": 2,
        "p_value": 8
    })
    .to_string(index=False)
)


# ============================================================
# 7. RANKING DESCRIPTIVE
# ============================================================

ranking = (
    hour_interaction[
        [
            "messages",
            "reads_rate_pct",
            "clicks_rate_pct",
            "replies_rate_pct",
            "any_engagement_rate_pct"
        ]
    ]
    .copy()
)

ranking["engagement_rank"] = (
    ranking["any_engagement_rate_pct"]
    .rank(ascending=False, method="min")
    .astype(int)
)

ranking = ranking.sort_values("engagement_rank")

print("\n" + "=" * 115)
print("DESCRIPTIVE RANKING — ANY ENGAGEMENT")
print("=" * 115)

print(
    ranking.round(2).to_string()
)

print(
    "\nNOTE: ranking is descriptive only; significance ≠ causality "
    "because historical send hour was not randomized."
)

SEND HOUR × INTERACTION — ALL MESSAGES
           messages  reads_rate_pct  clicks_rate_pct  replies_rate_pct  any_engagement_rate_pct
send_hour                                                                                      
9             10242           18.33             0.00              4.63                    22.95
10            10340           21.52             0.00              5.55                    27.07
11            10429           23.73             0.00              6.04                    29.77
12             8222           26.70             0.00              7.05                    33.75
13             5246           26.23             0.00              6.67                    32.90
14             5128           21.65             0.00              6.22                    27.87
15             5057           21.42             0.00              5.58                    26.99
16             5186           22.54             0.00              5.65                    28.19
1

In [29]:
# ============================================================
# CORRECTION — CLICKED_LINK
# ============================================================

hist["is_read"] = hist["interaction"].eq("read").astype(int)

hist["is_clicked"] = (
    hist["interaction"].eq("clicked_link")
).astype(int)

hist["is_replied"] = hist["interaction"].eq("replied").astype(int)

hist["any_engagement"] = (
    hist["interaction"]
    .isin(["read", "clicked_link", "replied"])
    .astype(int)
)

# ------------------------------------------------------------
# ALL MESSAGES
# ------------------------------------------------------------

hour_eng = (
    hist
    .groupby("send_hour")
    .agg(
        messages=("message_id", "count"),
        reads=("is_read", "sum"),
        clicks=("is_clicked", "sum"),
        replies=("is_replied", "sum"),
        engaged=("any_engagement", "sum")
    )
)

for col in ["reads", "clicks", "replies", "engaged"]:
    hour_eng[f"{col}_rate_pct"] = (
        hour_eng[col] / hour_eng["messages"] * 100
    )

print("=" * 100)
print("CORRECTED — ALL MESSAGES")
print("=" * 100)

print(
    hour_eng[
        [
            "messages",
            "reads_rate_pct",
            "clicks_rate_pct",
            "replies_rate_pct",
            "engaged_rate_pct"
        ]
    ].round(2).to_string()
)


# ------------------------------------------------------------
# GLOBAL TESTS
# ------------------------------------------------------------

from scipy.stats import chi2_contingency

print("\n" + "=" * 100)
print("GLOBAL TESTS")
print("=" * 100)

for name, col in [
    ("READ", "is_read"),
    ("CLICK", "is_clicked"),
    ("REPLY", "is_replied"),
    ("ANY ENGAGEMENT", "any_engagement")
]:
    
    tab = pd.crosstab(hist["send_hour"], hist[col])
    chi2, p, dof, _ = chi2_contingency(tab)

    print(
        f"{name:<15} "
        f"chi2={chi2:8.2f} | "
        f"dof={dof:2d} | "
        f"p={p:.8f}"
    )


# ------------------------------------------------------------
# DELIVERED ONLY
# ------------------------------------------------------------

deliv = hist.loc[
    hist["delivery_status"].eq("delivered")
].copy()

deliv_hour = (
    deliv
    .groupby("send_hour")
    .agg(
        delivered=("message_id", "count"),
        reads=("is_read", "sum"),
        clicks=("is_clicked", "sum"),
        replies=("is_replied", "sum"),
        engaged=("any_engagement", "sum")
    )
)

for col in ["reads", "clicks", "replies", "engaged"]:
    deliv_hour[f"{col}_rate_pct"] = (
        deliv_hour[col] / deliv_hour["delivered"] * 100
    )

print("\n" + "=" * 100)
print("CORRECTED — DELIVERED ONLY")
print("=" * 100)

print(
    deliv_hour[
        [
            "delivered",
            "reads_rate_pct",
            "clicks_rate_pct",
            "replies_rate_pct",
            "engaged_rate_pct"
        ]
    ].round(2).to_string()
)

CORRECTED — ALL MESSAGES
           messages  reads_rate_pct  clicks_rate_pct  replies_rate_pct  engaged_rate_pct
send_hour                                                                               
9             10242           18.33             8.63              4.63             31.59
10            10340           21.52             9.44              5.55             36.51
11            10429           23.73            11.07              6.04             40.84
12             8222           26.70            12.32              7.05             46.07
13             5246           26.23            11.28              6.67             44.19
14             5128           21.65            10.80              6.22             38.67
15             5057           21.42            10.32              5.58             37.31
16             5186           22.54            10.28              5.65             38.47
17             5196           23.61            11.41              6.12             41

In [30]:
# ============================================================
# ADJUSTED SEND-HOUR EFFECT ON ENGAGEMENT
# Controls: DPD + template + weekday
# Clustered SE by customer
# ============================================================

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# ------------------------------------------------------------
# 1. DATA
# ------------------------------------------------------------

adj = wa.copy()

adj["sent_at"] = pd.to_datetime(adj["sent_at"])

adj = adj.loc[
    (adj["sent_at"] < pd.Timestamp("2026-09-01"))
    & (adj["sent_at"].dt.hour.between(9, 20))
].copy()

adj["send_hour"] = adj["sent_at"].dt.hour
adj["weekday"] = adj["sent_at"].dt.day_name()

# Correct interaction flags
adj["read"] = adj["interaction"].eq("read").astype(int)
adj["click"] = adj["interaction"].eq("clicked_link").astype(int)
adj["reply"] = adj["interaction"].eq("replied").astype(int)

adj["engaged"] = (
    adj["interaction"]
    .isin(["read", "clicked_link", "replied"])
    .astype(int)
)

# DPD bucket
adj["dpd_bucket"] = pd.cut(
    adj["days_past_due"],
    bins=[0, 7, 15, 30, 45, 60, np.inf],
    labels=[
        "01–07",
        "08–15",
        "16–30",
        "31–45",
        "46–60",
        "60+"
    ],
    include_lowest=True
)

# ------------------------------------------------------------
# 2. MODEL SAMPLE
# ------------------------------------------------------------

model_df = adj[
    [
        "customer_id",
        "send_hour",
        "weekday",
        "template",
        "dpd_bucket",
        "engaged",
        "read",
        "click",
        "reply"
    ]
].dropna().copy()

# categorical strings
for col in [
    "send_hour",
    "weekday",
    "template",
    "dpd_bucket"
]:
    model_df[col] = model_df[col].astype(str)

print("=" * 105)
print("ADJUSTED HOUR × ENGAGEMENT")
print("=" * 105)

print(f"Rows      : {len(model_df):,}")
print(f"Customers : {model_df['customer_id'].nunique():,}")


# ============================================================
# 3. FUNCTION
# ============================================================

def fit_hour_model(df, outcome):

    formula = (
        f"{outcome} ~ "
        "C(send_hour, Treatment(reference='9')) + "
        "C(dpd_bucket) + "
        "C(template) + "
        "C(weekday)"
    )

    model = smf.logit(
        formula=formula,
        data=df
    ).fit(
        disp=False,
        cov_type="cluster",
        cov_kwds={
            "groups": df["customer_id"]
        }
    )

    # --------------------------------------------------------
    # Extract hour coefficients
    # --------------------------------------------------------

    rows = []

    for hour in range(10, 21):

        key = (
            "C(send_hour, Treatment(reference='9'))"
            f"[T.{hour}]"
        )

        beta = model.params.get(key, np.nan)
        se = model.bse.get(key, np.nan)
        p = model.pvalues.get(key, np.nan)

        OR = np.exp(beta)

        ci_low = np.exp(beta - 1.96 * se)
        ci_high = np.exp(beta + 1.96 * se)

        rows.append({
            "hour": hour,
            "OR_vs_09h": OR,
            "CI_low": ci_low,
            "CI_high": ci_high,
            "p_value": p
        })

    result = pd.DataFrame(rows)

    return model, result


# ============================================================
# 4. RUN MODELS
# ============================================================

results = {}

for outcome in [
    "engaged",
    "read",
    "click",
    "reply"
]:

    model, table = fit_hour_model(
        model_df,
        outcome
    )

    results[outcome] = {
        "model": model,
        "table": table
    }

    print("\n" + "=" * 105)
    print(outcome.upper())
    print("=" * 105)

    print(
        table
        .round({
            "OR_vs_09h": 3,
            "CI_low": 3,
            "CI_high": 3,
            "p_value": 6
        })
        .to_string(index=False)
    )


# ============================================================
# 5. PREDICTED ENGAGEMENT BY HOUR
#
# Standardization:
# keep the observed population exactly the same and change
# only send_hour for everybody.
# ============================================================

eng_model = results["engaged"]["model"]

predictions = []

for hour in range(9, 21):

    temp = model_df.copy()

    temp["send_hour"] = str(hour)

    p = eng_model.predict(temp)

    predictions.append({
        "hour": hour,
        "adjusted_engagement_pct": p.mean() * 100
    })

pred = pd.DataFrame(predictions)

baseline = (
    pred.loc[
        pred["hour"].eq(9),
        "adjusted_engagement_pct"
    ]
    .iloc[0]
)

pred["uplift_vs_09h_pp"] = (
    pred["adjusted_engagement_pct"]
    - baseline
)

print("\n" + "=" * 105)
print("STANDARDIZED ADJUSTED ENGAGEMENT")
print("=" * 105)

print(
    pred
    .round({
        "adjusted_engagement_pct": 2,
        "uplift_vs_09h_pp": 2
    })
    .to_string(index=False)
)


# ============================================================
# 6. 18–19h FOCUS
# ============================================================

print("\n" + "=" * 105)
print("FOCUS — 18h / 19h")
print("=" * 105)

for outcome in [
    "engaged",
    "read",
    "click",
    "reply"
]:

    tab = results[outcome]["table"]

    focus = tab.loc[
        tab["hour"].isin([18, 19])
    ]

    print(f"\n{outcome.upper()}")

    print(
        focus
        .round({
            "OR_vs_09h": 3,
            "CI_low": 3,
            "CI_high": 3,
            "p_value": 6
        })
        .to_string(index=False)
    )


print(
    "\nNOTE: adjusted historical association only. "
    "The historical send hour was not randomized, "
    "so this does not establish causality."
)

ADJUSTED HOUR × ENGAGEMENT
Rows      : 75,406
Customers : 11,724

ENGAGED
 hour  OR_vs_09h  CI_low  CI_high  p_value
   10       1.25    1.18     1.33     0.00
   11       1.50    1.41     1.58     0.00
   12       1.87    1.76     1.98     0.00
   13       1.72    1.60     1.84     0.00
   14       1.37    1.28     1.47     0.00
   15       1.29    1.20     1.39     0.00
   16       1.36    1.26     1.46     0.00
   17       1.52    1.42     1.63     0.00
   18       2.14    1.98     2.32     0.00
   19       2.05    1.89     2.21     0.00
   20       1.53    1.41     1.66     0.00

READ
 hour  OR_vs_09h  CI_low  CI_high  p_value
   10       1.23    1.15     1.32     0.00
   11       1.41    1.31     1.51     0.00
   12       1.65    1.53     1.77     0.00
   13       1.60    1.47     1.73     0.00
   14       1.23    1.13     1.34     0.00
   15       1.23    1.13     1.34     0.00
   16       1.30    1.20     1.42     0.00
   17       1.38    1.27     1.50     0.00
   18       1.84 

In [31]:
# ============================================================
# 18–19h vs OTHER HOURS — PAYMENT & RECOVERY
# ============================================================

import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
from statsmodels.stats.proportion import proportions_ztest

df = wa.copy()
df["sent_at"] = pd.to_datetime(df["sent_at"])

df = df.loc[
    (df["sent_at"] < pd.Timestamp("2026-09-01"))
    & (df["sent_at"].dt.hour.between(9, 20))
].copy()

df["send_hour"] = df["sent_at"].dt.hour

df["hour_group"] = np.where(
    df["send_hour"].isin([18, 19]),
    "18–19h",
    "Other hours"
)

# ------------------------------------------------------------
# Outcomes
# ------------------------------------------------------------

df["paid"] = df["paid_within_72h"].fillna(0).astype(int)

df["recovery"] = (
    df["amount_paid_brl"]
    .fillna(0)
    .clip(lower=0)
)

# ============================================================
# 1. DESCRIPTIVE
# ============================================================

summary = (
    df.groupby("hour_group")
      .agg(
          messages=("message_id", "count"),
          customers=("customer_id", "nunique"),
          payments=("paid", "sum"),
          recovered_brl=("recovery", "sum"),
          avg_balance=("outstanding_balance_brl", "mean")
      )
)

summary["payment_rate_pct"] = (
    summary["payments"] / summary["messages"] * 100
)

summary["recovery_per_msg"] = (
    summary["recovered_brl"] / summary["messages"]
)

print("=" * 100)
print("18–19h vs OTHER HOURS — ECONOMIC OUTCOMES")
print("=" * 100)

print(summary.round(2).to_string())


# ============================================================
# 2. PAYMENT RATE TEST
# H0: payment rate 18–19h = payment rate other hours
# ============================================================

a = df.loc[df["hour_group"].eq("18–19h")]
b = df.loc[df["hour_group"].eq("Other hours")]

count = np.array([
    a["paid"].sum(),
    b["paid"].sum()
])

nobs = np.array([
    len(a),
    len(b)
])

z, p_payment = proportions_ztest(
    count=count,
    nobs=nobs
)

rate_a = a["paid"].mean()
rate_b = b["paid"].mean()

print("\n" + "=" * 100)
print("PAYMENT RATE")
print("=" * 100)

print(f"18–19h      : {rate_a*100:.2f}%")
print(f"Other hours : {rate_b*100:.2f}%")
print(f"Difference  : {(rate_a-rate_b)*100:+.2f} p.p.")
print(f"Relative    : {(rate_a/rate_b-1)*100:+.1f}%")
print(f"Z-stat      : {z:.3f}")
print(f"P-value     : {p_payment:.8f}")


# ============================================================
# 3. RECOVERY / MESSAGE
#
# Includes zero recovery, because every message costs money
# and zero is an economically meaningful outcome.
# ============================================================

mean_a = a["recovery"].mean()
mean_b = b["recovery"].mean()

u, p_recovery = mannwhitneyu(
    a["recovery"],
    b["recovery"],
    alternative="two-sided"
)

print("\n" + "=" * 100)
print("RECOVERY PER MESSAGE")
print("=" * 100)

print(f"18–19h      : R$ {mean_a:,.2f}")
print(f"Other hours : R$ {mean_b:,.2f}")
print(f"Difference  : R$ {mean_a-mean_b:+,.2f}")
print(f"Relative    : {(mean_a/mean_b-1)*100:+.1f}%")
print(f"Mann-Whitney U : {u:,.0f}")
print(f"P-value        : {p_recovery:.8f}")


# ============================================================
# 4. RECOVERY AMONG PAYERS
#
# Diagnostic only:
# Is the difference coming from more people paying,
# or larger payments conditional on payment?
# ============================================================

pay_a = a.loc[a["recovery"] > 0, "recovery"]
pay_b = b.loc[b["recovery"] > 0, "recovery"]

u_pay, p_pay_amount = mannwhitneyu(
    pay_a,
    pay_b,
    alternative="two-sided"
)

print("\n" + "=" * 100)
print("RECOVERY CONDITIONAL ON PAYMENT")
print("=" * 100)

print(f"18–19h avg payer      : R$ {pay_a.mean():,.2f}")
print(f"Other hours avg payer : R$ {pay_b.mean():,.2f}")
print(f"18–19h median         : R$ {pay_a.median():,.2f}")
print(f"Other median          : R$ {pay_b.median():,.2f}")
print(f"P-value               : {p_pay_amount:.8f}")


# ============================================================
# 5. NET RECOVERY / MESSAGE
# R$1 cost per attempt
# ============================================================

summary["net_recovery_per_msg"] = (
    summary["recovery_per_msg"] - 1
)

print("\n" + "=" * 100)
print("NET RECOVERY / MESSAGE — R$1 MESSAGE COST")
print("=" * 100)

print(
    summary[
        [
            "messages",
            "payment_rate_pct",
            "recovery_per_msg",
            "net_recovery_per_msg"
        ]
    ]
    .round(2)
    .to_string()
)

print(
    "\nNOTE: this is still an observational comparison. "
    "A significant difference does not establish a causal hour effect."
)

18–19h vs OTHER HOURS — ECONOMIC OUTCOMES
             messages  customers  payments  recovered_brl  avg_balance  payment_rate_pct  recovery_per_msg
hour_group                                                                                                
18–19h           7212       5159       608     371,613.79       830.70              8.43             51.53
Other hours     68194      11614      5018   3,087,691.51       822.62              7.36             45.28

PAYMENT RATE
18–19h      : 8.43%
Other hours : 7.36%
Difference  : +1.07 p.p.
Relative    : +14.6%
Z-stat      : 3.295
P-value     : 0.00098507

RECOVERY PER MESSAGE
18–19h      : R$ 51.53
Other hours : R$ 45.28
Difference  : R$ +6.25
Relative    : +13.8%
Mann-Whitney U : 248,518,875
P-value        : 0.00111207

RECOVERY CONDITIONAL ON PAYMENT
18–19h avg payer      : R$ 611.21
Other hours avg payer : R$ 615.32
18–19h median         : R$ 483.50
Other median          : R$ 483.20
P-value               : 0.51277302

NET RECOVER

In [33]:
# ============================================================
# FINAL ADJUSTED TEST — ROBUST VERSION
# 18–19h vs OTHER HOURS
#
# Outcome: paid within 72h
# Controls:
#   DPD continuous
#   template
#   weekday
#   log(balance)
#
# GLM Binomial + clustered SE by customer
# ============================================================

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

# ------------------------------------------------------------
# 1. DATA
# ------------------------------------------------------------

df = wa.copy()

df["sent_at"] = pd.to_datetime(df["sent_at"])

df = df.loc[
    (df["sent_at"] < pd.Timestamp("2026-09-01"))
    & (df["sent_at"].dt.hour.between(9, 20))
].copy()

df["send_hour"] = df["sent_at"].dt.hour

df["prime_time"] = (
    df["send_hour"].isin([18, 19])
).astype(int)

df["weekday"] = df["sent_at"].dt.day_name()

df["paid"] = (
    df["paid_within_72h"]
    .fillna(0)
    .astype(int)
)

df["recovery"] = (
    df["amount_paid_brl"]
    .fillna(0)
    .clip(lower=0)
)

df["log_balance"] = np.log1p(
    df["outstanding_balance_brl"]
)

model_df = df[
    [
        "customer_id",
        "prime_time",
        "paid",
        "recovery",
        "days_past_due",
        "template",
        "weekday",
        "log_balance"
    ]
].dropna().copy()


print("=" * 100)
print("FINAL ADJUSTED TEST — 18–19h vs OTHER HOURS")
print("=" * 100)

print(f"Rows      : {len(model_df):,}")
print(f"Customers : {model_df['customer_id'].nunique():,}")
print(f"18–19h    : {model_df['prime_time'].sum():,}")
print(f"Others    : {(model_df['prime_time'] == 0).sum():,}")


# ============================================================
# 2. GLM BINOMIAL
# ============================================================

formula = """
paid ~
prime_time
+ days_past_due
+ C(template)
+ C(weekday)
+ log_balance
"""

payment_model = smf.glm(
    formula=formula,
    data=model_df,
    family=sm.families.Binomial()
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": model_df["customer_id"]
    }
)


# ============================================================
# 3. PRIME-TIME EFFECT
# ============================================================

beta = payment_model.params["prime_time"]
se = payment_model.bse["prime_time"]
p = payment_model.pvalues["prime_time"]

OR = np.exp(beta)

ci_low = np.exp(beta - 1.96 * se)
ci_high = np.exp(beta + 1.96 * se)

print("\n" + "=" * 100)
print("ADJUSTED PAYMENT EFFECT — 18–19h")
print("=" * 100)

print(f"Coefficient : {beta:.4f}")
print(f"Odds Ratio  : {OR:.3f}")
print(f"95% CI      : [{ci_low:.3f}, {ci_high:.3f}]")
print(f"P-value     : {p:.8f}")


# ============================================================
# 4. STANDARDIZED PROBABILITIES
#
# Same observed population.
# Only prime_time changes.
# ============================================================

other = model_df.copy()
prime = model_df.copy()

other["prime_time"] = 0
prime["prime_time"] = 1

p_other = payment_model.predict(other).mean()
p_prime = payment_model.predict(prime).mean()

diff_pp = (
    p_prime - p_other
) * 100

relative = (
    p_prime / p_other - 1
) * 100


print("\n" + "=" * 100)
print("STANDARDIZED ADJUSTED PAYMENT")
print("=" * 100)

print(f"Other hours : {p_other*100:.2f}%")
print(f"18–19h      : {p_prime*100:.2f}%")
print(f"Difference  : {diff_pp:+.2f} p.p.")
print(f"Relative    : {relative:+.1f}%")


# ============================================================
# 5. RAW COMPARISON
# ============================================================

raw = (
    model_df
    .groupby("prime_time")
    .agg(
        messages=("paid", "size"),
        payment_rate=("paid", "mean"),
        recovery_per_msg=("recovery", "mean")
    )
)

raw["payment_rate"] *= 100

raw.index = raw.index.map({
    0: "Other hours",
    1: "18–19h"
})

print("\n" + "=" * 100)
print("RAW vs ADJUSTED")
print("=" * 100)

print(raw.round(2).to_string())


# ============================================================
# 6. MODEL QA
# ============================================================

print("\n" + "=" * 100)
print("MODEL QA")
print("=" * 100)

print(f"Converged : {payment_model.converged}")
print(f"Deviance  : {payment_model.deviance:,.2f}")
print(f"Pearson χ²: {payment_model.pearson_chi2:,.2f}")

print(
    "\nInterpretation: adjusted historical association only; "
    "historical send hour was not randomized."
)

FINAL ADJUSTED TEST — 18–19h vs OTHER HOURS
Rows      : 75,406
Customers : 11,724
18–19h    : 7,212
Others    : 68,194

ADJUSTED PAYMENT EFFECT — 18–19h
Coefficient : 0.1504
Odds Ratio  : 1.162
95% CI      : [1.064, 1.270]
P-value     : 0.00088051

STANDARDIZED ADJUSTED PAYMENT
Other hours : 7.36%
18–19h      : 8.44%
Difference  : +1.08 p.p.
Relative    : +14.7%

RAW vs ADJUSTED
             messages  payment_rate  recovery_per_msg
prime_time                                           
Other hours     68194          7.36             45.28
18–19h           7212          8.43             51.53

MODEL QA
Converged : True
Deviance  : 39,435.31
Pearson χ²: 75,735.43

Interpretation: adjusted historical association only; historical send hour was not randomized.
